# NB05b v29 -- Dataset Builder (v2 Parquet Architecture)

> Copyright (C) 2024-2026 Marco Heinzen - SPDX-License-Identifier: AGPL-3.0-or-later
> Part of the Master Thesis "Building Damage Assessment with Multimodal Satellite Time Series and Machine Learning in the Russia-Ukraine War 2022-2026"
> Code hosted at https://github.com/marcoheinzen/bda
> Parts of this code were written or improved with the assistance of Claude (Anthropic); all other code, and the concept, research, architecture, design, execution, testing and validation throughout, are the author's work.


## v29 changes vs v28 (final V2 methodological fix)

**Aggregation rules now statistic-aware per product family** (replaces universal mean+std):

- Per-scene raw (A1, A2, A3, A5, A6), composite raw (A9), accumulator extrema (A14, A19, A20, A21), prepost CARD (A13): full set `mean, p10, p50, p90, std, min, max, max_abs_delta`
- Per-scene categorical (A4) and composite categorical (A10): `mode_raw, mode_corrected` (with veg-removal rule + 0.80 emergency exit), all class fractions, `urban_or_bare_frac`
- Composite vs scene delta (A11) + prepost CARD delta (A13 delta): `mean, p10, p50, p90, std, min, max, max_abs_delta`
- Accumulator counts/rates (drop_count, rise_count, exceedance_count): `mean, max, min`
- Block stats (A15) + rolling stats (A16, A17, A18): per-stat `mean, p10, p90, mean_abs_delta, min, max`
- All accumulator date columns: METADATA (added to META_COLS_SET) — `min_nonzero, max, count_unique`

**6 new atomic parquets for NB03e v47 R2b/P1d outputs** (consume new `temporal/{COH,CARD,MS}/{rolling_accum,block_accum}/` directories):
- A23 `rolling_accum_coh`, A24 `rolling_accum_card`, A25 `rolling_accum_ms`
- A26 `block_accum_coh`, A27 `block_accum_card`, A28 `block_accum_ms`

**No deletion of any cells.** A7/A8 rolling means kept as historical artifacts (methodologically inferior to accumulators per matched-filter principle); thesis ablation will compare A7/A8 vs A23/A24/A25.

**Strict data/metadata taxonomy:** date columns, scenes_observed, n_pixels, was_observed flags, calendar dates → metadata. Pre-baseline mean/std, urban_retained, loss_fraction, mode_*, lu_at_*, modal_*, final_class → features.

## Atomic parquets (architectural notes)

- Parquet schema unchanged — same column-naming convention `{prefix}__{stat}`
- Tier system stays
- Metadata leakage prevention is downstream NB07/NB09 .py module concern: NB05b writes everything, manifest tags every column with `role` ∈ {id, label, metadata, feature}, downstream filters by role tag
 
**bda** -- Building Damage Assessment using satellite imagery> Copyright (C) 2024-2026 Marco Heinzen> SPDX-License-Identifier: AGPL-3.0-or-later>> This program is free software: you can redistribute it and/or modify> it under the terms of the GNU Affero General Public License as published by> the Free Software Foundation, either version 3 of the License, or> (at your option) any later version.>> This program is distributed in the hope that it will be useful,> but WITHOUT ANY WARRANTY; without even the implied warranty of> MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the> GNU Affero General Public License for more details.>> You should have received a copy of the GNU Affero General Public License> along with this program. If not, see <https://www.gnu.org/licenses/>.| Field | Value ||---|---|| Notebook | `05bV2_bda_Dataset_Builder_v20.ipynb` || Version | `v20` || Pipeline position | After NB03e/NB04a (products+QA), before NB06/NB07/NB09. Creates v2/ parquets. || Repository | [github.com/marcoheinzen/bda](https://github.com/marcoheinzen/bda) || License | Code: AGPL-3.0-or-later -- see [LICENSE](../LICENSE) || Last validated | `2026-04-17` |## v2 Parquet Architecture**Parquet = experiment definition.** Each parquet is a self-contained testable hypothesis.NB09 loads a parquet and trains on ALL its feature columns. No column selection at experiment time.- **18 atomic** + **8 fusion** = **26 parquets per tier**- All tracked in `parquet_manifest.json`- NaN handling happens once here, not at experiment time- No class balancing -- natural prevalence everywhere- GroupKFold by city is non-negotiable### Inputs- `data_stack/{city}/` TIF products from NB03d/NB03e- `data_stack/{city}/building_labels.tif` from NB05a- `data_stack/{city}/buildings_overture_with_damage.geojson`### Outputs- `DATASET_ROOT/v2/bda_*.parquet` (26 per tier)- `DATASET_ROOT/v2/parquet_manifest.json`- `DATASET_ROOT/v2/groupkfold.parquet`

In [1]:
# @title CELL 1: NB05b v2 CONFIG
TIER_SELECTION = [0]
CITY_SELECTION = None
REQUIRE_UNOSAT = True
MIN_PIXELS = 3

# per-parquet force rerun (False = skip if parquet exists and is valid)
FR_BUILDINGS = False
FR_SCENE_MS = True
FR_SCENE_CARD = True
FR_SCENE_COH = True
FR_SCENE_LANDUSE = True         # A4 — parquet absent on disk → builds regardless
FR_SCENE_INDICES = True
FR_SCENE_NBR = True             # A6 — parquet absent on disk → builds regardless
FR_ROLLING_COH = True
FR_ROLLING_CARD = True
FR_COMPOSITE_PREPOST_BANDS = True
FR_COMPOSITE_PREPOST_LANDUSE = True   # <-- CHANGE: A10 already built correctly
FR_COMPOSITE_VS_SCENES_BANDS = True   # <-- CHANGE: A11 already built correctly
FR_COMPOSITE_VS_SCENES_LANDUSE = True # A12 — parquet absent on disk → builds regardless
FR_PREPOST_SINGLE_CARD = True
FR_COH_DROP = True
FR_CARD_DROP = True
FR_MS_CHANGE = True
FR_MS_MAHA = True
FR_LU_CHANGE = True
FR_BLOCK_STATS = True
FR_ROLLING_STATS = True
# v29 additions: NB03e v47 R2b/P1d output parquets
FR_ROLLING_ACCUM_COH = True
FR_ROLLING_ACCUM_CARD = True
FR_ROLLING_ACCUM_MS = True
FR_BLOCK_ACCUM_COH = True
FR_BLOCK_ACCUM_CARD = True
FR_BLOCK_ACCUM_MS = True
FR_FUSIONS = True
FR_GROUPKFOLD = True
FR_MANIFEST = True

In [2]:
# @title CELL 2: LOAD GLOBAL SETUP
import platform, os
if platform.system() == 'Windows':
    _setup = r'F:\PROJECTS\masterthesis\gdrive\masterthesis\notebooks\global_setup.py'
elif os.path.exists('/content/drive_f'):
    _setup = '/content/drive_f/masterthesis/notebooks/global_setup.py'
else:
    _setup = '/mnt/f/PROJECTS/masterthesis/gdrive/masterthesis/notebooks/global_setup.py'
with open(_setup) as f:
    exec(f.read())

BDA GLOBAL SETUP
Started: 2026-04-26 16:10:11
Python: 3.12.12

[1/7] Directory Structure
----------------------------------------------------------------------


/home/alpineobotics/miniconda3/envs/bda/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


  GDrive (G:):       /content/drive_f/masterthesis OK
  GDrive (F:):       /content/drive_f/masterthesis OK
  Local data (G:):   /content/masterthesis_local/data OK
  Data stack (F:):   /mnt/f/PROJECTS/masterthesis/data_stack OK

  TIER_SELECTION: [0]
  CITY_SELECTION: None (tier filter)
  REQUIRE_UNOSAT: True
  CITIES_TO_PROCESS: 4 cities
    Lysychansk (battle_start=2022-06-25)
    Mariupol (battle_start=2022-02-24)
    Rubizhne (battle_start=2022-03-04)
    Sievierodonetsk (battle_start=2022-05-05)

[2/7] Credentials
----------------------------------------------------------------------
  Copernicus: inf***
  OpenTopography: OK
  Earthdata: marcoheinzen

[3/7] Python Packages
----------------------------------------------------------------------


<string>:564: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.



  Already installed: 23
  Newly installed:   0
  Failed:            0

[4/7] Global Imports & Configuration
----------------------------------------------------------------------
  All imports loaded

[5/7] Processing Config & SNAP
----------------------------------------------------------------------
  GPT: Usage:
  Temporal baseline: 10-24 days
  Wavelength: 0.0555

[6/7] GPU Status
----------------------------------------------------------------------
  CUDA available: NVIDIA GeForce RTX 2070 SUPER
    CUDA version: 12.8

[7/7] Disk Space
----------------------------------------------------------------------
  GDrive (G:)     910.2/7452.0 GB (6541.8 GB free)
  GDrive (F:)     1384.5/3726.0 GB (2341.5 GB free)
  Local data      11560.0/14901.9 GB (3341.8 GB free)
  Data stack      1384.5/3726.0 GB (2341.5 GB free)
  WSL ext4        68.6/1006.9 GB (887.1 GB free)

GLOBAL SETUP COMPLETE
  Torch device: cuda
  Cities: 4, CITY=Lysychansk
  Functions: load_aoi(), load_aoi_gdf(), load_aoi

# SHARED: Load helpers + discover cities (grouped by tier)

In [3]:
# @title CELL 3: SHARED HELPERS + CITY DISCOVERY (PER-TIER)
import sys, importlib, re, time, gc, json
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from pathlib import Path
from datetime import datetime

# ---- v2 output directory ----
# DATASET_ROOT_V2 derived from STACK_DIR (always clean, never includes v1/v2)
# Mirrors DATASET_ROOT_V1 from global_setup but for v2 parquets
DATASET_ROOT_V2 = STACK_DIR / 'dataset' / 'v2'
DATASET_ROOT_V2.mkdir(parents=True, exist_ok=True)
V2_DIR = DATASET_ROOT_V2  # alias used throughout this notebook
print(f"  DATASET_ROOT_V1: {DATASET_ROOT_V1}")
print(f"  DATASET_ROOT_V2: {DATASET_ROOT_V2}")

# ---- v2 parquet path helpers ----
def v2_path(name, tier):
    return V2_DIR / f"bda_{name}_t{tier}.parquet"

from stack_catalog import BDACatalog, parse_feature_name
cat = BDACatalog(CATALOG_DB)
print(f"  Catalog: {CATALOG_DB}")

import stack_features
importlib.reload(stack_features)
from stack_features import extract_zonal_stats_from_labels  # legacy, used only as fallback

import aggregation_helpers
importlib.reload(aggregation_helpers)
from aggregation_helpers import (
    extract_zonal_aware, extract_zonal_categorical, extract_zonal_dates,
    date_metadata_columns, DEFAULT_MIN_PIXELS,
)
aggregation_helpers._print_banner()

# ---- discover ML-ready cities ----
manifest_path = STACK_ROOT / "data_stack_manifest.json"
with open(manifest_path) as f:
    manifest = json.load(f)

city_meta = {}
for city_name, info in sorted(manifest.get("cities", {}).items()):
    tier = info.get("tier", 99)
    if TIER_SELECTION and tier not in TIER_SELECTION:
        continue
    if CITY_SELECTION and city_name not in (CITY_SELECTION if isinstance(CITY_SELECTION, list) else [CITY_SELECTION]):
        continue
    if not info.get("ready_ml", False):
        continue
    ref_path = STACK_ROOT / city_name / "reference_grid.json"
    if not ref_path.exists():
        continue
    aoi_row = load_aoi(city_name)
    city_meta[city_name] = {
        "battle_start": str(aoi_row.get("battle_start", ""))[:10],
        "battle_stop": str(aoi_row.get("battle_stop", "") or "")[:10],
        "tier": int(aoi_row.get("tier", 99)),
        "manifest": info,
    }

CITIES = sorted(city_meta.keys())

# ---- GROUP CITIES BY TIER ----
from collections import defaultdict
TIER_CITIES = defaultdict(list)
for c in CITIES:
    TIER_CITIES[city_meta[c]['tier']].append(c)
TIER_CITIES = dict(sorted(TIER_CITIES.items()))

print(f"Cities: {len(CITIES)} across {len(TIER_CITIES)} tiers")
for tier, tc in TIER_CITIES.items():
    print(f"  Tier {tier}: {len(tc)} cities")
    for c in tc:
        m = city_meta[c]
        print(f"    {c:<22s} battle={m['battle_start']}")

# ---- helper: load building labels + metadata for one city ----
def load_city_buildings(city_name):
    city_stack = STACK_ROOT / city_name
    labels_path = city_stack / "building_labels.tif"
    meta_path = city_stack / "building_raster_meta.json"
    if not labels_path.exists():
        return None, None, None, None
    with rasterio.open(labels_path) as src:
        label_array = src.read(1)
        ref_shape = (src.height, src.width)
    with open(meta_path) as f:
        bldg_meta = json.load(f)
    n_buildings = bldg_meta["n_buildings"]
    return label_array, ref_shape, n_buildings, bldg_meta

# ---- helper: assign period label from date + battle dates ----
def get_period_label(date_str, battle_start, battle_stop):
    d_clean = str(date_str).replace('-', '')[:8]
    dt = datetime.strptime(d_clean, '%Y%m%d')
    bs = datetime.strptime(str(battle_start)[:10], '%Y-%m-%d')
    if dt < bs:
        return 'prebattle'
    be = None
    if battle_stop and str(battle_stop).lower() not in ('', 'none', 'nat', 'ongoing'):
        be = datetime.strptime(str(battle_stop)[:10], '%Y-%m-%d')
    if be and dt > be:
        return 'postbattle'
    return 'crossbattle'

# ---- helper: assign timestep index ----
def assign_timesteps(dates, battle_start_str):
    bs = datetime.strptime(battle_start_str, "%Y-%m-%d")
    dated = [(d, datetime.strptime(d, "%Y%m%d")) for d in dates]
    pre = sorted([(d, dt) for d, dt in dated if dt < bs], key=lambda x: x[1])
    post = sorted([(d, dt) for d, dt in dated if dt >= bs], key=lambda x: x[1])
    ts = {}
    for i, (d, dt) in enumerate(pre):
        ts[d] = -(len(pre) - i)
    for i, (d, dt) in enumerate(post):
        ts[d] = i
    return ts

# ---- helper: read + pad + nodata-mask a TIF ----
def read_tif_aligned(path, ref_shape):
    with rasterio.open(path) as src:
        data = src.read(1).astype(np.float32)
        if data.shape != ref_shape:
            padded = np.full(ref_shape, np.nan, dtype=np.float32)
            h, w = min(data.shape[0], ref_shape[0]), min(data.shape[1], ref_shape[1])
            padded[:h, :w] = data[:h, :w]
            data = padded
        nd = src.nodata
        if nd is not None:
            try:
                nd_f = float(nd)
                if not np.isnan(nd_f):
                    data[data == nd_f] = np.nan
            except (ValueError, TypeError):
                pass
    return data


# ---- COLUMN ROLE DEFINITIONS (source of truth for all parquets) ----
ID_COLS = {'building_id'}
LABEL_COLS = {'damage_binary', 'damage_label', 'damage', 'ems98_grade'}
META_COLS_SET = {
    'city', 'tier', 'battle_start', 'battle_stop', 'conflict_ongoing',
    'has_card', 'has_coh', 'has_ms',
    'unosat_id', 'unosat_date', 'unosat_ep',
    'match_method', 'match_distance', 'in_aoi',
    'height', 'num_floors', 'roof_height',
    'area_m2', 'centroid_x', 'centroid_y', 'n_pixels',
    'date', 'date1', 'date2', 'timestep', 'period_label',
    'was_observed_ms', 'was_observed_card', 'was_observed_coh',
    'was_observed_cohdrop', 'was_observed_blockstats',
    'pre_date_card', 'post_date_card', 'dataset',
    # --- v29 additions: date columns from accumulators (battle-calendar leakage) ---
    # date columns from coh_drop, card_drop, ms_change, ms_maha, lu_change accumulators
    # patterns covered: {prefix}__date_first_*__{min_nonzero|max|count_unique|n_pixels_valid}
    # patterns covered: {prefix}__date_worst_*__{min_nonzero|max|count_unique|n_pixels_valid}
    # patterns covered: {prefix}__date_persistent_*__{min_nonzero|max|count_unique|n_pixels_valid}
    # patterns covered: scenes_observed (orbit-availability dependent, operational metadata)
    # rather than enumerating each, add a regex-ish helper at the bottom of this cell;
    # builders also add their date columns explicitly below as they are produced.
}
META_DATE_PATTERNS = (
    'date_first_drop', 'date_worst_drop',
    'date_first_exceedance', 'date_worst_exceedance',
    'date_first_swir_rise', 'date_worst_swir_rise',
    'date_first_loss', 'date_persistent_loss',
    'pre_date_ms', 'post_date_ms', 'pre_date_coh', 'post_date_coh',
)
META_OPERATIONAL_PATTERNS = (
    'scenes_observed', 'n_pixels_valid',
)

def _is_metadata_column(col):
    """True if a column should be treated as metadata, not feature.

    Strict V2 rule: any column whose value depends on calendar / battle
    timeline / orbit availability / footprint geometry is metadata.
    """
    if col in META_COLS_SET:
        return True
    if col in ID_COLS or col in LABEL_COLS:
        return True
    for pat in META_DATE_PATTERNS:
        if pat in col:
            return True
    for pat in META_OPERATIONAL_PATTERNS:
        if col.endswith('__' + pat) or col == pat:
            return True
    return False

print('  META taxonomy: META_COLS_SET + META_DATE_PATTERNS + META_OPERATIONAL_PATTERNS, _is_metadata_column() ready')

def build_role_overrides(columns):
    overrides = {}
    for col in columns:
        if col in ID_COLS:
            overrides[col] = 'id'
        elif col in LABEL_COLS:
            overrides[col] = 'label'
        elif _is_metadata_column(col):
            overrides[col] = 'metadata'
    return overrides

print("  Roles: ID_COLS, LABEL_COLS, META_COLS_SET, build_role_overrides()")

# ---- helper: log dataset profile to JSON for NB06 discovery ----
PROFILE_DIR = V2_DIR / 'dataset_profiles'
PROFILE_DIR.mkdir(parents=True, exist_ok=True)

def log_dataset_profile(df, parquet_name):
    from datetime import datetime as _dt
    num_cols = [c for c in df.columns if df[c].dtype.kind in ('f', 'i', 'u')]
    feat_cols = [c for c in num_cols if not _is_metadata_column(c)]

    profile = {
        'parquet_name': parquet_name,
        'timestamp': _dt.now().isoformat(),
        'n_rows': int(len(df)),
        'n_cols': int(len(df.columns)),
        'n_features': len(feat_cols),
        'n_cities': int(df['city'].nunique()) if 'city' in df.columns else 0,
        'nan_rate_overall': float(df[feat_cols].isna().mean().mean()) if feat_cols else 0.0,
    }

    if 'city' in df.columns:
        profile['buildings_per_city'] = {
            c: int(n) for c, n in df['city'].value_counts().items()
        }

    if 'city' in df.columns and 'damage_binary' in df.columns:
        dr = df.groupby('city')['damage_binary'].mean()
        profile['damage_rate_per_city'] = {c: float(v) for c, v in dr.items()}

    if 'city' in df.columns and feat_cols:
        nan_per_city = df.groupby('city')[feat_cols].apply(
            lambda g: g.isna().mean().mean()
        )
        profile['nan_rate_per_city'] = {c: float(v) for c, v in nan_per_city.items()}

    out = PROFILE_DIR / f'{parquet_name}.json'
    with open(out, 'w') as f:
        json.dump(profile, f, indent=2)

    from scipy import stats as _stats
    feat_records = []
    for col in feat_cols:
        s = df[col]
        n_nan = int(s.isna().sum())
        nan_pct = 100.0 * n_nan / len(s) if len(s) > 0 else 0.0
        valid = s.dropna()
        rec = {'feature': col, 'nan_pct': round(nan_pct, 2)}
        if len(valid) > 1:
            m = float(valid.mean())
            sd = float(valid.std())
            rec['mean'] = round(m, 6)
            rec['std'] = round(sd, 6)
            rec['min'] = round(float(valid.min()), 6)
            rec['max'] = round(float(valid.max()), 6)
            rec['cv'] = round(sd / abs(m), 4) if abs(m) > 1e-12 else 0.0
            rec['skewness'] = round(float(_stats.skew(valid, nan_policy='omit')), 4)
            rec['kurtosis'] = round(float(_stats.kurtosis(valid, nan_policy='omit')), 4)
        else:
            rec.update({'mean': None, 'std': None, 'min': None, 'max': None,
                        'cv': None, 'skewness': None, 'kurtosis': None})
        feat_records.append(rec)

    feat_out = PROFILE_DIR / f'{parquet_name}_features.json'
    with open(feat_out, 'w') as f:
        json.dump(feat_records, f, indent=1)
    print(f"    profile -> {out.name} + {feat_out.name} ({len(feat_records)} features)")


# ---- helper: global median imputation + was_observed flags for wide parquets (Plan Section 6.2) ----
def impute_wide_parquet(df, modality_name):
    """Apply global median imputation to a wide parquet.
    
    1. Adds was_observed_{modality_name} flag (1 if building has any non-NaN feature, 0 otherwise)
    2. Fills remaining NaN with global (cross-city) median per column
    3. Returns the modified DataFrame
    
    Plan Section 6.2: "Period-aggregated wide parquets: global median imputation
    per feature column + binary was_observed flag per modality group."
    """
    feat_cols = [c for c in df.columns
                 if not _is_metadata_column(c) and df[c].dtype.kind in ('f', 'i', 'u')]
    
    if not feat_cols:
        return df
    
    # was_observed: 1 if building has at least one non-NaN feature value
    flag_col = f"was_observed_{modality_name}"
    df[flag_col] = df[feat_cols].notna().any(axis=1).astype(int)
    
    # global median imputation (across all cities, not per-city to avoid leakage)
    medians = df[feat_cols].median()
    n_before = df[feat_cols].isna().sum().sum()
    df[feat_cols] = df[feat_cols].fillna(medians)
    n_after = df[feat_cols].isna().sum().sum()
    
    print(f"    impute_wide_parquet({modality_name}): {n_before} NaN -> {n_after} NaN, {flag_col} added")
    return df

print("  impute_wide_parquet() ready")

print(f"  PROFILE_DIR: {PROFILE_DIR}")

# ---- helper: save parquet with catalog registration ----
def save_v2_parquet(df, name, tier):
    out_path = v2_path(name, tier)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out_path, index=False)
    pq_name = f"bda_{name}_t{tier}"
    cat.register_parquet_columns(pq_name, df.columns, role_overrides=build_role_overrides(df.columns))
    log_dataset_profile(df, pq_name)
    return out_path

# ---- manifest accumulator ----
MANIFEST_ENTRIES = {}

def register_manifest(name, pq_id, fmt, join_keys, feature_columns, question, tier, n_rows, depends_on_tifs=None, composed_of=None, cities_excluded=None):
    _prev_excl = MANIFEST_ENTRIES.get(name, {}).get('cities_excluded_per_tier', {})
    MANIFEST_ENTRIES[name] = {
        'id': pq_id,
        'pattern': f"v2/bda_{name}_t{{tier}}.parquet",
        'format': fmt,
        'join_keys': join_keys,
        'feature_columns': feature_columns,
        'n_features': len(feature_columns),
        'experiment_question': question,
        'tiers_built': MANIFEST_ENTRIES.get(name, {}).get('tiers_built', []) + [tier],
        'n_rows_per_tier': {**MANIFEST_ENTRIES.get(name, {}).get('n_rows_per_tier', {}), str(tier): n_rows},
    }
    if depends_on_tifs:
        MANIFEST_ENTRIES[name]['depends_on_tifs'] = depends_on_tifs
    if composed_of:
        MANIFEST_ENTRIES[name]['composed_of'] = composed_of
    if cities_excluded is not None:
        MANIFEST_ENTRIES[name]['cities_excluded_per_tier'] = {**_prev_excl, str(tier): sorted(list(cities_excluded))}
    elif _prev_excl:
        MANIFEST_ENTRIES[name]['cities_excluded_per_tier'] = _prev_excl

print("  Manifest accumulator ready")


# ==== BUGFIX: disk-scan manifest registration ====
# MANIFEST_ENTRIES is populated only by cells that actually run; skipped cells
# (parquet exists + FR_*=False) never register. Same issue across tier runs.
# This static catalog + scan_disk_and_register() fills MANIFEST_ENTRIES from
# every on-disk bda_*_t*.parquet before the manifest write, without re-running
# any of the heavy builders.

PARQUET_CATALOG = {
    'buildings':                   {'id': 'A0',  'format': 'wide', 'join_keys': ['city', 'building_id'],         'question': 'Building metadata table'},
    'scene_ms':                    {'id': 'A1',  'format': 'long', 'join_keys': ['city', 'building_id', 'date'], 'question': 'Q1: Does optical carry per-scene damage signal?'},
    'scene_card':                  {'id': 'A2',  'format': 'long', 'join_keys': ['city', 'building_id', 'date'], 'question': 'Q1: Does CARD carry per-scene damage signal?'},
    'scene_coh':                   {'id': 'A3',  'format': 'long', 'join_keys': ['city', 'building_id', 'date'], 'question': 'Q1: Does COH carry per-scene damage signal?'},
    'scene_landuse':               {'id': 'A4',  'format': 'long', 'join_keys': ['city', 'building_id', 'date'], 'question': 'Q5: Does landuse context help?'},
    'scene_indices':               {'id': 'A5',  'format': 'long', 'join_keys': ['city', 'building_id', 'date'], 'question': 'Q1: Do spectral indices outperform raw bands?'},
    'scene_nbr':                   {'id': 'A6',  'format': 'long', 'join_keys': ['city', 'building_id', 'date'], 'question': 'Q1: Does NBR carry signal?'},
    'rolling_coh':                 {'id': 'A7',  'format': 'long', 'join_keys': ['city', 'building_id', 'date'], 'question': 'Q4: Do rolling windows add value (COH)?'},
    'rolling_card':                {'id': 'A8',  'format': 'long', 'join_keys': ['city', 'building_id', 'date'], 'question': 'Q4: Do rolling windows add value (CARD)?'},
    'composite_prepost_bands':     {'id': 'A9',  'format': 'wide', 'join_keys': ['city', 'building_id'],         'question': 'Q3: Do composites work? (Dietrich baseline)'},
    'composite_prepost_landuse':   {'id': 'A10', 'format': 'wide', 'join_keys': ['city', 'building_id'],         'question': 'Q5: Does composite landuse change help?'},
    'composite_vs_scenes_bands':   {'id': 'A11', 'format': 'long', 'join_keys': ['city', 'building_id', 'date'], 'question': 'Q3: Does per-scene trajectory beat static snapshot?'},
    'composite_vs_scenes_landuse': {'id': 'A12', 'format': 'long', 'join_keys': ['city', 'building_id', 'date'], 'question': 'Q5: Per-scene landuse trajectory?'},
    'prepost_single_card':         {'id': 'A13', 'format': 'wide', 'join_keys': ['city', 'building_id'],         'question': 'Q3: Single-scene CARD pre/post'},
    'coh_drop':                    {'id': 'A14', 'format': 'wide', 'join_keys': ['city', 'building_id'],         'question': 'Q1: Does cumulative COH drop carry signal?'},
    'block_stats':                 {'id': 'A15', 'format': 'wide', 'join_keys': ['city', 'building_id'],         'question': 'Q6: Does Dietrich block approach work with GroupKFold?'},
    'rolling_stats_roll3':         {'id': 'A16', 'format': 'wide', 'join_keys': ['city', 'building_id'],         'question': 'Q4: Rolling stats (window=3)'},
    'rolling_stats_roll7':         {'id': 'A17', 'format': 'wide', 'join_keys': ['city', 'building_id'],         'question': 'Q4: Rolling stats (window=7)'},
    'rolling_stats_roll13':        {'id': 'A18', 'format': 'wide', 'join_keys': ['city', 'building_id'],         'question': 'Q4: Rolling stats (window=13)'},
    'card_drop':                   {'id': 'A19', 'format': 'wide', 'join_keys': ['city', 'building_id'],         'question': 'Q: Does cumulative CARD z-score drop carry signal?'},
    'ms_change':                   {'id': 'A20', 'format': 'wide', 'join_keys': ['city', 'building_id'],         'question': 'Q: Does cumulative MS SWIR brightness + NBR anomaly carry signal?'},
    'ms_maha':                     {'id': 'A21', 'format': 'wide', 'join_keys': ['city', 'building_id'],         'question': 'Q: Does multi-band MS Mahalanobis distance carry signal?'},
    'lu_change':                   {'id': 'A22', 'format': 'wide', 'join_keys': ['city', 'building_id'],         'question': 'Q: Does persistent urban-to-other landuse loss carry signal?'},
    'fusion_ms_card':              {'id': 'F1',  'format': 'long', 'join_keys': ['city', 'building_id', 'date'], 'question': 'Q2: Does MS+CARD beat either alone?',        'composed_of': ['scene_ms', 'scene_card']},
    'fusion_ms_card_cohdrop':      {'id': 'F2',  'format': 'long', 'join_keys': ['city', 'building_id', 'date'], 'question': 'Q2: Full multimodal',                       'composed_of': ['scene_ms', 'scene_card', 'coh_drop']},
    'fusion_card_cohdrop':         {'id': 'F3',  'format': 'long', 'join_keys': ['city', 'building_id', 'date'], 'question': 'Q2: SAR-only multimodal',                   'composed_of': ['scene_card', 'coh_drop']},
    'fusion_ms_cohdrop':           {'id': 'F4',  'format': 'long', 'join_keys': ['city', 'building_id', 'date'], 'question': 'Q2: Optical + COH drop',                    'composed_of': ['scene_ms', 'coh_drop']},
    'fusion_indices_card':         {'id': 'F5',  'format': 'long', 'join_keys': ['city', 'building_id', 'date'], 'question': 'Q2: Indices + CARD',                        'composed_of': ['scene_indices', 'scene_card']},
    'fusion_indices_card_cohdrop': {'id': 'F6',  'format': 'long', 'join_keys': ['city', 'building_id', 'date'], 'question': 'Q2: Indices + CARD + COH drop',             'composed_of': ['scene_indices', 'scene_card', 'coh_drop']},
    'fusion_composite_cohdrop':    {'id': 'F7',  'format': 'wide', 'join_keys': ['city', 'building_id'],         'question': 'Q2: Composite + COH drop',                  'composed_of': ['composite_prepost_bands', 'coh_drop']},
    'fusion_composite_blockstats': {'id': 'F8',  'format': 'wide', 'join_keys': ['city', 'building_id'],         'question': 'Q6: Dietrich composite + block replication', 'composed_of': ['composite_prepost_bands', 'block_stats']},
    # --- v29 additions: NB03e v47 R2b/P1d output parquets ---
    'rolling_accum_coh':           {'id': 'A23', 'format': 'long', 'join_keys': ['city', 'building_id', 'date'], 'question': 'Q4b: Rolling-window matched-filter signal (COH)?'},
    'rolling_accum_card':          {'id': 'A24', 'format': 'long', 'join_keys': ['city', 'building_id', 'date'], 'question': 'Q4b: Rolling-window matched-filter signal (CARD)?'},
    'rolling_accum_ms':            {'id': 'A25', 'format': 'long', 'join_keys': ['city', 'building_id', 'date'], 'question': 'Q4b: Rolling-window matched-filter signal (MS)?'},
    'block_accum_coh':             {'id': 'A26', 'format': 'wide', 'join_keys': ['city', 'building_id'],         'question': 'Q6b: Block-scope matched-filter signal (COH)?'},
    'block_accum_card':            {'id': 'A27', 'format': 'wide', 'join_keys': ['city', 'building_id'],         'question': 'Q6b: Block-scope matched-filter signal (CARD)?'},
    'block_accum_ms':              {'id': 'A28', 'format': 'wide', 'join_keys': ['city', 'building_id'],         'question': 'Q6b: Block-scope matched-filter signal (MS)?'},
}

def scan_disk_and_register(verbose=True):
    """Walk V2_DIR for bda_*_t*.parquet and register any (name, tier) missing
    from MANIFEST_ENTRIES. Idempotent: runtime registrations win, disk-scan
    only fills gaps. Reads columns/rowcount via pyarrow metadata (no load)."""
    import pyarrow.parquet as _pq
    pat = re.compile(r'^bda_(.+)_t(\d+)\.parquet$')
    n_added = 0
    n_skip_registered = 0
    n_skip_unknown = 0
    for p in sorted(V2_DIR.glob('bda_*_t*.parquet')):
        m = pat.match(p.name)
        if not m:
            continue
        name = m.group(1)
        tier = int(m.group(2))
        existing = MANIFEST_ENTRIES.get(name)
        if existing and tier in existing.get('tiers_built', []):
            n_skip_registered += 1
            continue
        if name not in PARQUET_CATALOG:
            if verbose:
                print(f"  disk-scan: {p.name} -- no catalog entry, skipping")
            n_skip_unknown += 1
            continue
        meta = PARQUET_CATALOG[name]
        try:
            pf = _pq.ParquetFile(p)
            columns = pf.schema_arrow.names
            n_rows = pf.metadata.num_rows
        except Exception as e:
            if verbose:
                print(f"  disk-scan: {p.name} -- read error: {e}")
            continue
        feat_cols = [c for c in columns if not _is_metadata_column(c)]
        register_manifest(
            name,
            meta['id'],
            meta['format'],
            meta['join_keys'],
            feat_cols,
            meta['question'],
            tier,
            n_rows,
            composed_of=meta.get('composed_of'),
            depends_on_tifs=meta.get('depends_on_tifs'),
        )
        n_added += 1
        if verbose:
            print(f"  disk-scan: registered {p.name} ({n_rows} rows, {len(feat_cols)} features)")
    if verbose:
        print(f"  disk-scan summary: +{n_added} added, {n_skip_registered} already-registered, {n_skip_unknown} unknown-name")
    return n_added

print(f"  PARQUET_CATALOG: {len(PARQUET_CATALOG)} entries, scan_disk_and_register() ready")


  DATASET_ROOT_V1: /mnt/f/PROJECTS/masterthesis/data_stack/dataset/v1
  DATASET_ROOT_V2: /mnt/f/PROJECTS/masterthesis/data_stack/dataset/v2
  Catalog: /mnt/f/PROJECTS/masterthesis/data_stack/bda.sqlite
  aggregation_helpers loaded (v29.1 pandas-fast):
    extract_zonal_aware (pandas groupby)
    extract_zonal_categorical (np.bincount)
    extract_zonal_dates (pandas groupby, int32 throughout)
    DEFAULT_MIN_PIXELS=3
    VEG_OVERRIDE_THRESHOLD=0.2
    VEG_HONEST_THRESHOLD=0.8
Cities: 4 across 1 tiers
  Tier 0: 4 cities
    Lysychansk             battle=2022-06-25
    Mariupol               battle=2022-02-24
    Rubizhne               battle=2022-03-04
    Sievierodonetsk        battle=2022-05-05
  META taxonomy: META_COLS_SET + META_DATE_PATTERNS + META_OPERATIONAL_PATTERNS, _is_metadata_column() ready
  Roles: ID_COLS, LABEL_COLS, META_COLS_SET, build_role_overrides()
  impute_wide_parquet() ready
  PROFILE_DIR: /mnt/f/PROJECTS/masterthesis/data_stack/dataset/v2/dataset_profiles
  Man

# CELL 4: BUILDINGS METADATA PARQUET (per tier)

In [4]:
# @title CELL 4: bda_buildings_t{tier}.parquet
for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('buildings', tier)
    if not FR_BUILDINGS and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  bda_buildings_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier}: {len(tier_cities)} cities ---")

    GEOJSON_KEEP_COLS = {
        'damage', 'damage_label', 'damage_binary', 'ems98_grade',
        'unosat_id', 'unosat_date', 'unosat_ep',
        'match_method', 'match_distance', 'in_aoi',
        'height', 'num_floors', 'roof_height',
        'geometry',
    }
    NUMERIC_CAST = ['damage_binary', 'damage_label', 'damage', 'ems98_grade',
                    'height', 'num_floors', 'roof_height', 'match_distance', 'in_aoi']

    all_bldg = []
    for city_name in tier_cities:
        city_stack = STACK_ROOT / city_name
        bldg_path = city_stack / "buildings_overture_with_damage.geojson"
        if not bldg_path.exists():
            bldg_path = city_stack / "buildings_overture.geojson"
        if not bldg_path.exists():
            continue

        ref_path = city_stack / "reference_grid.json"
        with open(ref_path) as f:
            ref = json.load(f)

        gdf = gpd.read_file(bldg_path)
        keep = [c for c in GEOJSON_KEEP_COLS if c in gdf.columns]
        gdf = gdf[keep].copy()
        gdf = gdf.to_crs(f"EPSG:{ref['utm_epsg']}")

        gdf['city'] = city_name
        gdf['building_id'] = [f"{city_name}_{i}" for i in range(len(gdf))]
        gdf['area_m2'] = gdf.geometry.area
        gdf['centroid_x'] = gdf.geometry.centroid.x
        gdf['centroid_y'] = gdf.geometry.centroid.y

        for col in NUMERIC_CAST:
            if col in gdf.columns:
                gdf[col] = pd.to_numeric(gdf[col], errors='coerce')

        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is not None:
            pixel_counts = np.bincount(label_array.ravel(), minlength=n_buildings + 1)[1:]
            if len(pixel_counts) == len(gdf):
                gdf['n_pixels'] = pixel_counts
            else:
                print(f"  WARNING: {city_name} n_pixels mismatch: building_labels has {len(pixel_counts)} buildings, geojson has {len(gdf)}. Skipping n_pixels.")

        meta = city_meta[city_name]
        gdf['tier'] = meta['tier']
        gdf['battle_start'] = meta['battle_start']
        gdf['battle_stop'] = meta['battle_stop']
        conflict_ongoing = meta['battle_stop'] in ('', 'ongoing', None, 'None', 'NaT')
        gdf['conflict_ongoing'] = conflict_ongoing

        info = meta['manifest']
        gdf['has_card'] = bool(info.get('has_card', False))
        gdf['has_coh'] = bool(info.get('has_coh', False))
        gdf['has_ms'] = bool(info.get('has_ms', False))

        gdf_flat = gdf.drop(columns=['geometry'], errors='ignore')
        all_bldg.append(gdf_flat)
        print(f"  {city_name}: {len(gdf)} buildings")

    if all_bldg:
        df_bldg = pd.concat(all_bldg, ignore_index=True)
        save_v2_parquet(df_bldg, 'buildings', tier)
        print(f"  Saved: bda_buildings_t{tier} ({len(df_bldg)} buildings, {len(df_bldg.columns)} cols, {time.time()-t0:.0f}s)")


  bda_buildings_t0: exists (7.2 MB), skip


# CELL 5: A1 -- bda_scene_ms (per-scene optical, long-format)

In [5]:
# @title CELL 5: bda_scene_ms_t{tier}.parquet
MS_BANDS = ['b02', 'b03', 'b04', 'b05', 'b07', 'b08', 'b11', 'b12', 'b8a']
MS_AUX = ['scl', 'cloud_mask', 'visibility']

for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('scene_ms', tier)
    if not FR_SCENE_MS and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  bda_scene_ms_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} MS ---")
    rows = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue
        meta = city_meta[city_name]
        ms_dir = STACK_ROOT / city_name / "multispectral" / "flat"
        if not ms_dir.exists():
            continue

        dates = set()
        for f in ms_dir.glob("s2__b02__*.tif"):
            m = re.search(r'__(\d{8})\.tif$', f.name)
            if m:
                dates.add(m.group(1))

        ts_map = assign_timesteps(sorted(dates), meta['battle_start'])

        for date_str in sorted(dates):
            feats = {}
            for band in MS_BANDS:
                path = ms_dir / f"s2__{band}__{date_str}.tif"
                if not path.exists():
                    continue
                data = read_tif_aligned(path, ref_shape)
                stats = extract_zonal_aware(label_array, data, f"s2__{band}", n_buildings, agg="raw", min_pixels=MIN_PIXELS)
                feats.update(stats)

            for aux in MS_AUX:
                path = ms_dir / f"s2__{aux}__{date_str}.tif"
                if not path.exists():
                    continue
                data = read_tif_aligned(path, ref_shape)
                is_cat = aux == 'scl'
                # v29: aux bands - scl is categorical (mode helper), others raw.
                if is_cat:
                    stats = extract_zonal_categorical(label_array, data.astype("int32"), f"s2__{aux}", n_buildings, min_pixels=MIN_PIXELS)
                else:
                    stats = extract_zonal_aware(label_array, data, f"s2__{aux}", n_buildings, agg="raw", min_pixels=MIN_PIXELS)
                feats.update(stats)

            # --- fire flags (from landuse/flat/) ---
            fire_dir = STACK_ROOT / city_name / "landuse" / "flat"
            if fire_dir.exists():
                for fire_product in ['fire__active_fire', 'fire__burn_scar']:
                    fire_path = fire_dir / f"{fire_product}__{date_str}.tif"
                    if fire_path.exists():
                        data = read_tif_aligned(fire_path, ref_shape)
                        stats = extract_zonal_aware(label_array, data, fire_product, n_buildings, agg="raw", min_pixels=MIN_PIXELS)
                        feats.update(stats)

            if not feats:
                continue

            period = get_period_label(date_str, meta['battle_start'], meta['battle_stop'])
            timestep = ts_map.get(date_str, 0)
            bids = [f"{city_name}_{i}" for i in range(n_buildings)]

            date_df = pd.DataFrame(feats)
            date_df['building_id'] = bids
            date_df['city'] = city_name
            date_df['date'] = date_str
            date_df['timestep'] = timestep
            date_df['period_label'] = period
            rows.append(date_df)

        print(f"  {city_name}: {len(dates)} MS dates")

    if rows:
        df = pd.concat(rows, ignore_index=True)
        out = save_v2_parquet(df, 'scene_ms', tier)
        feat_cols = [c for c in df.columns if c not in (ID_COLS | LABEL_COLS | META_COLS_SET)]
        register_manifest('scene_ms', 'A1', 'long', ['city', 'building_id', 'date'], feat_cols,
                         'Q1: Does optical carry per-scene damage signal?', tier, len(df),
                         depends_on_tifs=['s2__b{XX}__{YYYYMMDD}.tif'])
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No MS scenes for tier {tier}")

    gc.collect()



--- Tier 0 MS ---
  Lysychansk: 11 MS dates
  Mariupol: 12 MS dates
  Rubizhne: 10 MS dates
  Sievierodonetsk: 12 MS dates
    profile -> bda_scene_ms_t0.json + bda_scene_ms_t0_features.json (97 features)
  Saved: bda_scene_ms_t0.parquet (2284825 rows, 114 cols, 97s)


# CELL 6: A2 -- bda_scene_card (per-scene CARD backscatter, long-format)

In [6]:
# @title CELL 6: bda_scene_card_t{tier}.parquet
for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('scene_card', tier)
    if not FR_SCENE_CARD and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  bda_scene_card_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} CARD ---")
    rows = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue
        meta = city_meta[city_name]
        card_dir = STACK_ROOT / city_name / "SAR_CARD" / "flat"
        if not card_dir.exists():
            continue

        dates = set()
        for f in card_dir.glob("s1__vv__*.tif"):
            m = re.search(r'__(\d{8})\.tif$', f.name)
            if m:
                dates.add(m.group(1))

        ts_map = assign_timesteps(sorted(dates), meta['battle_start'])

        for date_str in sorted(dates):
            feats = {}
            for pol in ['vv', 'vh']:
                path = card_dir / f"s1__{pol}__{date_str}.tif"
                if not path.exists():
                    candidates = list(card_dir.glob(f"s1__{pol}__o*__{date_str}.tif"))
                    if candidates:
                        path = candidates[0]
                    else:
                        continue
                data = read_tif_aligned(path, ref_shape)
                stats = extract_zonal_aware(label_array, data, f"s1__{pol}", n_buildings, agg="raw", min_pixels=MIN_PIXELS)
                feats.update(stats)

            if not feats:
                continue

            period = get_period_label(date_str, meta['battle_start'], meta['battle_stop'])
            timestep = ts_map.get(date_str, 0)
            bids = [f"{city_name}_{i}" for i in range(n_buildings)]

            date_df = pd.DataFrame(feats)
            date_df['building_id'] = bids
            date_df['city'] = city_name
            date_df['date'] = date_str
            date_df['timestep'] = timestep
            date_df['period_label'] = period
            rows.append(date_df)

        print(f"  {city_name}: {len(dates)} dates")

    if rows:
        df = pd.concat(rows, ignore_index=True)
        out = save_v2_parquet(df, 'scene_card', tier)
        feat_cols = [c for c in df.columns if c not in (ID_COLS | LABEL_COLS | META_COLS_SET)]
        register_manifest('scene_card', 'A2', 'long', ['city', 'building_id', 'date'], feat_cols,
                         'Q1: Does CARD carry per-scene damage signal?', tier, len(df),
                         depends_on_tifs=['s1__vv__{YYYYMMDD}.tif', 's1__vh__{YYYYMMDD}.tif'])
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No CARD scenes for tier {tier}")

    gc.collect()



--- Tier 0 CARD ---
  Lysychansk: 8 dates
  Mariupol: 14 dates
  Rubizhne: 13 dates
  Sievierodonetsk: 12 dates
    profile -> bda_scene_card_t0.json + bda_scene_card_t0_features.json (16 features)
  Saved: bda_scene_card_t0.parquet (2431593 rows, 23 cols, 25s)


# CELL 7: A3 -- bda_scene_coh (per-scene coherence, long-format)

In [7]:
# @title CELL 7: bda_scene_coh_t{tier}.parquet
for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('scene_coh', tier)
    if not FR_SCENE_COH and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  bda_scene_coh_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} COH ---")
    rows = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue
        meta = city_meta[city_name]
        coh_dir = STACK_ROOT / city_name / "SAR_SLC" / "flat"
        zscore_dir = STACK_ROOT / city_name / "temporal" / "COH" / "zscore"
        if not coh_dir.exists():
            continue

        coh_pairs = {}
        for f in sorted(coh_dir.glob("*.tif")):
            m = re.match(r's1__coh_(vv|vh)__(?:o\d{3}__)?(\d{8})_(\d{8})\.tif$', f.name)
            if m:
                pol, d1, d2 = m.group(1), m.group(2), m.group(3)
                key = (d1, d2)
                coh_pairs.setdefault(key, {})[pol] = f
                continue
            m = re.match(r'COH_(VV|VH)_(PRE|CROSS|POST)_(?:o\d{3}_)?(\d{8})_(\d{8})\.tif$', f.name)
            if m:
                pol, period_tag, d1, d2 = m.group(1).lower(), m.group(2), m.group(3), m.group(4)
                key = (d1, d2)
                coh_pairs.setdefault(key, {})[pol] = f
                coh_pairs[key]['_period_tag'] = period_tag
                continue

        zscore_map = {}
        if zscore_dir.exists():
            for f in sorted(zscore_dir.glob("*.tif")):
                m = re.search(r'__(\d{8})\.tif$', f.name)
                if m:
                    zscore_map[m.group(1)] = f

        pair_dates = sorted(set(d2 for (d1, d2) in coh_pairs.keys()))
        ts_map = assign_timesteps(pair_dates, meta['battle_start'])

        for (d1, d2), pol_dict in sorted(coh_pairs.items()):
            feats = {}
            for pol in ['vv', 'vh']:
                path = pol_dict.get(pol)
                if path is None or not path.exists():
                    continue
                data = read_tif_aligned(path, ref_shape)
                stats = extract_zonal_aware(label_array, data, f"s1__coh_{pol}", n_buildings, agg="raw", min_pixels=MIN_PIXELS)
                feats.update(stats)

            zs_path = zscore_map.get(d2)
            if zs_path and zs_path.exists():
                data = read_tif_aligned(zs_path, ref_shape)
                stats = extract_zonal_aware(label_array, data, "s1__coh_vv__zscore", n_buildings, agg="raw", min_pixels=MIN_PIXELS)
                feats.update(stats)

            if not feats:
                continue

            period = get_period_label(d2, meta['battle_start'], meta['battle_stop'])
            timestep = ts_map.get(d2, 0)
            bids = [f"{city_name}_{i}" for i in range(n_buildings)]

            date_df = pd.DataFrame(feats)
            date_df['building_id'] = bids
            date_df['city'] = city_name
            date_df['date1'] = d1
            date_df['date2'] = d2
            date_df['date'] = d2
            date_df['timestep'] = timestep
            date_df['period_label'] = period
            rows.append(date_df)

        print(f"  {city_name}: {len(coh_pairs)} COH pairs")

    if rows:
        df = pd.concat(rows, ignore_index=True)
        out = save_v2_parquet(df, 'scene_coh', tier)
        feat_cols = [c for c in df.columns if c not in (ID_COLS | LABEL_COLS | META_COLS_SET)]
        register_manifest('scene_coh', 'A3', 'long', ['city', 'building_id', 'date'], feat_cols,
                         'Q1: Does COH carry per-scene damage signal?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No COH scenes for tier {tier}")

    gc.collect()



--- Tier 0 COH ---
  Lysychansk: 5 COH pairs
  Mariupol: 7 COH pairs
  Rubizhne: 5 COH pairs
  Sievierodonetsk: 6 COH pairs
    profile -> bda_scene_coh_t0.json + bda_scene_coh_t0_features.json (24 features)
  Saved: bda_scene_coh_t0.parquet (1237935 rows, 34 cols, 14s)


# CELL 8: A4 -- bda_scene_landuse (per-scene landuse, long-format)

In [8]:
# @title CELL 8: bda_scene_landuse_t{tier}.parquet
DATE_RE = re.compile(r'__(\d{8})\.tif$')

for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('scene_landuse', tier)
    if not FR_SCENE_LANDUSE and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  bda_scene_landuse_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} LANDUSE ---")
    rows = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue
        meta = city_meta[city_name]
        flat_dir = STACK_ROOT / city_name / "landuse" / "flat"
        if not flat_dir.exists():
            continue

        date_files = {}
        for tif in flat_dir.glob("s2__landuse__*.tif"):
            m = DATE_RE.search(tif.name)
            if m:
                date_files.setdefault(m.group(1), []).append(tif)

        ts_map = assign_timesteps(sorted(date_files.keys()), meta['battle_start'])

        for date_str in sorted(date_files.keys()):
            feats = {}
            for tif in date_files[date_str]:
                data = read_tif_aligned(tif, ref_shape)
                stats = extract_zonal_categorical(label_array, data.astype("int32"), "s2__landuse", n_buildings, min_pixels=MIN_PIXELS)
                feats.update(stats)

            if not feats:
                continue

            period = get_period_label(date_str, meta['battle_start'], meta['battle_stop'])
            timestep = ts_map.get(date_str, 0)
            bids = [f"{city_name}_{i}" for i in range(n_buildings)]

            date_df = pd.DataFrame(feats)
            date_df['building_id'] = bids
            date_df['city'] = city_name
            date_df['date'] = date_str
            date_df['timestep'] = timestep
            date_df['period_label'] = period
            rows.append(date_df)

        print(f"  {city_name}: {len(date_files)} landuse dates")

    if rows:
        df = pd.concat(rows, ignore_index=True)
        out = save_v2_parquet(df, 'scene_landuse', tier)
        feat_cols = [c for c in df.columns if c not in (ID_COLS | LABEL_COLS | META_COLS_SET)]
        register_manifest('scene_landuse', 'A4', 'long', ['city', 'building_id', 'date'], feat_cols,
                         'Q5: Does landuse context help?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No landuse scenes for tier {tier}")

    gc.collect()



--- Tier 0 LANDUSE ---
  Lysychansk: 14 landuse dates
  Mariupol: 12 landuse dates
  Rubizhne: 13 landuse dates
  Sievierodonetsk: 15 landuse dates
    profile -> bda_scene_landuse_t0.json + bda_scene_landuse_t0_features.json (9 features)
  Saved: bda_scene_landuse_t0.parquet (2518846 rows, 15 cols, 8s)


# CELL 9: A5 -- bda_scene_indices (per-scene spectral indices, long-format)

In [9]:
# @title CELL 9: bda_scene_indices_t{tier}.parquet
KNOWN_INDICES = ['ndvi', 'bsi', 'savi', 'mndwi', 'ndbi', 'ndsi', 'ibi', 'baei', 'ui']
DATE_RE = re.compile(r'__(\d{8})\.tif$')

for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('scene_indices', tier)
    if not FR_SCENE_INDICES and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  bda_scene_indices_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} INDICES ---")
    rows = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue
        meta = city_meta[city_name]
        flat_dir = STACK_ROOT / city_name / "landuse" / "flat"
        if not flat_dir.exists():
            continue

        date_files = {}
        for idx_name in KNOWN_INDICES:
            for tif in flat_dir.glob(f"s2__{idx_name}__*.tif"):
                m = DATE_RE.search(tif.name)
                if m:
                    date_files.setdefault(m.group(1), []).append(tif)
            for tif in flat_dir.glob(f"{idx_name}__*.tif"):
                m = DATE_RE.search(tif.name)
                if m:
                    date_files.setdefault(m.group(1), []).append(tif)

        ts_map = assign_timesteps(sorted(date_files.keys()), meta['battle_start'])

        for date_str in sorted(date_files.keys()):
            feats = {}
            for tif in date_files[date_str]:
                stem = tif.stem
                base = stem[:-(len(date_str)+2)]
                if base.startswith('s2__'):
                    prefix = base
                elif base in KNOWN_INDICES:
                    prefix = f"s2__{base}"
                else:
                    prefix = base
                data = read_tif_aligned(tif, ref_shape)
                stats = extract_zonal_aware(label_array, data, prefix, n_buildings, agg="raw", min_pixels=MIN_PIXELS)
                feats.update(stats)

            if not feats:
                continue

            period = get_period_label(date_str, meta['battle_start'], meta['battle_stop'])
            timestep = ts_map.get(date_str, 0)
            bids = [f"{city_name}_{i}" for i in range(n_buildings)]

            date_df = pd.DataFrame(feats)
            date_df['building_id'] = bids
            date_df['city'] = city_name
            date_df['date'] = date_str
            date_df['timestep'] = timestep
            date_df['period_label'] = period
            rows.append(date_df)

        print(f"  {city_name}: {len(date_files)} indices dates")

    if rows:
        df = pd.concat(rows, ignore_index=True)
        out = save_v2_parquet(df, 'scene_indices', tier)
        feat_cols = [c for c in df.columns if c not in (ID_COLS | LABEL_COLS | META_COLS_SET)]
        register_manifest('scene_indices', 'A5', 'long', ['city', 'building_id', 'date'], feat_cols,
                         'Q1: Do spectral indices outperform raw bands?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No index scenes for tier {tier}")

    gc.collect()



--- Tier 0 INDICES ---
  Lysychansk: 14 indices dates
  Mariupol: 12 indices dates
  Rubizhne: 13 indices dates
  Sievierodonetsk: 15 indices dates
    profile -> bda_scene_indices_t0.json + bda_scene_indices_t0_features.json (72 features)
  Saved: bda_scene_indices_t0.parquet (2518846 rows, 86 cols, 113s)


# CELL 10: A6 -- bda_scene_nbr (per-scene NBR, long-format)

In [10]:
# @title CELL 10: bda_scene_nbr_t{tier}.parquet
for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('scene_nbr', tier)
    if not FR_SCENE_NBR and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  bda_scene_nbr_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} NBR ---")
    rows = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue
        meta = city_meta[city_name]
        ms_dir = STACK_ROOT / city_name / "landuse" / "flat"
        if not ms_dir.exists():
            continue

        dates = set()
        for f in ms_dir.glob("s2__nbr__*.tif"):
            m = re.search(r'__(\d{8})\.tif$', f.name)
            if m:
                dates.add(m.group(1))

        ts_map = assign_timesteps(sorted(dates), meta['battle_start'])

        for date_str in sorted(dates):
            path = ms_dir / f"s2__nbr__{date_str}.tif"
            if not path.exists():
                continue
            data = read_tif_aligned(path, ref_shape)
            stats = extract_zonal_aware(label_array, data, "s2__nbr", n_buildings, agg="raw", min_pixels=MIN_PIXELS)
            if not stats:
                continue

            period = get_period_label(date_str, meta['battle_start'], meta['battle_stop'])
            timestep = ts_map.get(date_str, 0)
            bids = [f"{city_name}_{i}" for i in range(n_buildings)]

            date_df = pd.DataFrame(stats)
            date_df['building_id'] = bids
            date_df['city'] = city_name
            date_df['date'] = date_str
            date_df['timestep'] = timestep
            date_df['period_label'] = period
            rows.append(date_df)

        print(f"  {city_name}: {len(dates)} NBR dates")

    if rows:
        df = pd.concat(rows, ignore_index=True)
        out = save_v2_parquet(df, 'scene_nbr', tier)
        feat_cols = [c for c in df.columns if c not in (ID_COLS | LABEL_COLS | META_COLS_SET)]
        register_manifest('scene_nbr', 'A6', 'long', ['city', 'building_id', 'date'], feat_cols,
                         'Q1: Does NBR carry signal?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No NBR scenes for tier {tier}")

    gc.collect()



--- Tier 0 NBR ---
  Lysychansk: 14 NBR dates
  Mariupol: 12 NBR dates
  Rubizhne: 13 NBR dates
  Sievierodonetsk: 15 NBR dates
    profile -> bda_scene_nbr_t0.json + bda_scene_nbr_t0_features.json (8 features)
  Saved: bda_scene_nbr_t0.parquet (2518846 rows, 14 cols, 15s)


# CELL 11: A7 -- bda_rolling_coh (all windows, long-format)

In [11]:
# @title CELL 11: bda_rolling_coh_t{tier}.parquet
ROLL_WINDOWS = ROLLING_WINDOW_SIZES  # [3, 7, 13]

for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('rolling_coh', tier)
    if not FR_ROLLING_COH and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  bda_rolling_coh_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} ROLLING COH ---")
    rows = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue
        meta = city_meta[city_name]
        roll_dir = STACK_ROOT / city_name / "temporal" / "COH" / "rolling"
        if not roll_dir.exists():
            continue

        all_dates = set()
        for ws in ROLL_WINDOWS:
            for f in roll_dir.glob(f"s1__coh_vv__roll{ws}__*.tif"):
                m = re.search(r'__(\d{8})\.tif$', f.name)
                if m:
                    all_dates.add(m.group(1))

        ts_map = assign_timesteps(sorted(all_dates), meta['battle_start'])

        for date_str in sorted(all_dates):
            feats = {}
            for ws in ROLL_WINDOWS:
                path = roll_dir / f"s1__coh_vv__roll{ws}__{date_str}.tif"
                if not path.exists():
                    continue
                data = read_tif_aligned(path, ref_shape)
                stats = extract_zonal_aware(label_array, data, f"s1__coh_vv__roll{ws}", n_buildings, agg="raw", min_pixels=MIN_PIXELS)
                feats.update(stats)

            if not feats:
                continue

            period = get_period_label(date_str, meta['battle_start'], meta['battle_stop'])
            timestep = ts_map.get(date_str, 0)
            bids = [f"{city_name}_{i}" for i in range(n_buildings)]

            date_df = pd.DataFrame(feats)
            date_df['building_id'] = bids
            date_df['city'] = city_name
            date_df['date'] = date_str
            date_df['timestep'] = timestep
            date_df['period_label'] = period
            rows.append(date_df)

        print(f"  {city_name}: {len(all_dates)} rolling COH dates")

    if rows:
        df = pd.concat(rows, ignore_index=True)
        out = save_v2_parquet(df, 'rolling_coh', tier)
        feat_cols = [c for c in df.columns if c not in (ID_COLS | LABEL_COLS | META_COLS_SET)]
        register_manifest('rolling_coh', 'A7', 'long', ['city', 'building_id', 'date'], feat_cols,
                         'Q4: Do rolling windows add value (COH)?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No rolling COH for tier {tier}")

    gc.collect()



--- Tier 0 ROLLING COH ---
  Lysychansk: 3 rolling COH dates
  Mariupol: 5 rolling COH dates
  Rubizhne: 3 rolling COH dates
  Sievierodonetsk: 4 rolling COH dates
    profile -> bda_rolling_coh_t0.json + bda_rolling_coh_t0_features.json (16 features)
  Saved: bda_rolling_coh_t0.parquet (843707 rows, 23 cols, 6s)


# CELL 12: A8 -- bda_rolling_card (all windows, long-format)

In [12]:
# @title CELL 12: bda_rolling_card_t{tier}.parquet
for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('rolling_card', tier)
    if not FR_ROLLING_CARD and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  bda_rolling_card_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} ROLLING CARD ---")
    rows = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue
        meta = city_meta[city_name]
        roll_dir = STACK_ROOT / city_name / "temporal" / "CARD" / "rolling"
        if not roll_dir.exists():
            continue

        all_dates = set()
        for ws in ROLL_WINDOWS:
            for f in roll_dir.glob(f"s1__vv__roll{ws}__*.tif"):
                m = re.search(r'__(\d{8})\.tif$', f.name)
                if m:
                    all_dates.add(m.group(1))

        ts_map = assign_timesteps(sorted(all_dates), meta['battle_start'])

        for date_str in sorted(all_dates):
            feats = {}
            for ws in ROLL_WINDOWS:
                for pol in ['vv', 'vh']:
                    path = roll_dir / f"s1__{pol}__roll{ws}__{date_str}.tif"
                    if not path.exists():
                        continue
                    data = read_tif_aligned(path, ref_shape)
                    stats = extract_zonal_aware(label_array, data, f"s1__{pol}__roll{ws}", n_buildings, agg="raw", min_pixels=MIN_PIXELS)
                    feats.update(stats)

            if not feats:
                continue

            period = get_period_label(date_str, meta['battle_start'], meta['battle_stop'])
            timestep = ts_map.get(date_str, 0)
            bids = [f"{city_name}_{i}" for i in range(n_buildings)]

            date_df = pd.DataFrame(feats)
            date_df['building_id'] = bids
            date_df['city'] = city_name
            date_df['date'] = date_str
            date_df['timestep'] = timestep
            date_df['period_label'] = period
            rows.append(date_df)

        print(f"  {city_name}: {len(all_dates)} rolling CARD dates")

    if rows:
        df = pd.concat(rows, ignore_index=True)
        out = save_v2_parquet(df, 'rolling_card', tier)
        feat_cols = [c for c in df.columns if c not in (ID_COLS | LABEL_COLS | META_COLS_SET)]
        register_manifest('rolling_card', 'A8', 'long', ['city', 'building_id', 'date'], feat_cols,
                         'Q4: Do rolling windows add value (CARD)?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No rolling CARD for tier {tier}")

    gc.collect()



--- Tier 0 ROLLING CARD ---
  Lysychansk: 6 rolling CARD dates
  Mariupol: 12 rolling CARD dates
  Rubizhne: 11 rolling CARD dates
  Sievierodonetsk: 10 rolling CARD dates
    profile -> bda_rolling_card_t0.json + bda_rolling_card_t0_features.json (48 features)
  Saved: bda_rolling_card_t0.parquet (2037365 rows, 59 cols, 36s)


# CELL 13: A9 -- bda_composite_prepost_bands (wide, pre vs post composite)

In [13]:
# @title CELL 13: bda_composite_prepost_bands_t{tier}.parquet
# Wide-format: one row per building. MS composite bands only (no CARD/COH - those are in A15).
# Columns: {band}__{period}__{stat} + delta columns.
# Sources: composites/{period}/*.tif (excluding landuse TIFs which go to A10)

for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('composite_prepost_bands', tier)
    if not FR_COMPOSITE_PREPOST_BANDS and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  composite_prepost_bands_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} COMPOSITE PREPOST BANDS ---")
    all_city_dfs = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue

        feats = {}
        bids = [f"{city_name}_{i}" for i in range(n_buildings)]

        # --- MS composites ONLY (per period, no landuse) ---
        comp_dir = STACK_ROOT / city_name / "multispectral" / "composites"
        if not comp_dir.exists():
            continue

        for period_dir in sorted(comp_dir.iterdir()):
            if not period_dir.is_dir():
                if period_dir.suffix == '.tif' and 'landuse' not in period_dir.stem.lower():
                    prefix = period_dir.stem
                    data = read_tif_aligned(period_dir, ref_shape)
                    stats = extract_zonal_aware(label_array, data, prefix, n_buildings, agg="composite_raw", min_pixels=MIN_PIXELS)
                    feats.update(stats)
                continue

            period = period_dir.name
            for tif in sorted(period_dir.glob("*.tif")):
                stem = tif.stem
                if 'landuse' in stem.lower():
                    continue  # landuse goes to A10

                if stem.startswith('qa__'):
                    col_prefix = f"{stem}__{period}"
                elif stem.startswith(('s2__', 'composite_')):
                    band = stem.replace('composite_', '')
                    col_prefix = f"s2__{band}__{period}" if not stem.startswith('s2__') else f"{stem}__{period}"
                elif stem.startswith('visibility_'):
                    col_prefix = f"vis__{stem}__{period}"
                else:
                    col_prefix = f"{period}__{stem}"

                data = read_tif_aligned(tif, ref_shape)
                stats = extract_zonal_aware(label_array, data, col_prefix, n_buildings, agg="composite_raw", min_pixels=MIN_PIXELS)
                feats.update(stats)

        if not feats:
            continue

        city_df = pd.DataFrame(feats)
        city_df['building_id'] = bids
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)

        # compute delta columns
        delta_count = 0
        pre_patterns = [
            ('__prebattle_baseline__', '__post_winter_baseline__'),
            ('__winter_baseline__', '__post_winter_baseline__'),
        ]
        for pre_tag, post_tag in pre_patterns:
            pre_cols = [c for c in df.columns if pre_tag in c and c.endswith('_mean')]
            for pre_col in pre_cols:
                post_col = pre_col.replace(pre_tag, post_tag)
                if post_col in df.columns:
                    delta_name = pre_col.replace(pre_tag, '__delta__')
                    if delta_name not in df.columns:
                        df[delta_name] = df[post_col] - df[pre_col]
                        delta_count += 1

        # NaN imputation (Plan Section 6.2)
        df = impute_wide_parquet(df, 'composite')

        out = save_v2_parquet(df, 'composite_prepost_bands', tier)
        feat_cols = [c for c in df.columns if c not in (ID_COLS | LABEL_COLS | META_COLS_SET)]
        register_manifest('composite_prepost_bands', 'A9', 'wide', ['city', 'building_id'], feat_cols,
                         'Q3: Do composites work? (Dietrich baseline)', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {delta_count} deltas, {time.time()-t0:.0f}s)")
    else:
        print(f"  No composite data for tier {tier}")

    gc.collect()



--- Tier 0 COMPOSITE PREPOST BANDS ---
  Lysychansk: 567 feature columns
  Mariupol: 567 feature columns
  Rubizhne: 567 feature columns
  Sievierodonetsk: 567 feature columns
    impute_wide_parquet(composite): 26188978 NaN -> 0 NaN, was_observed_composite added
    profile -> bda_composite_prepost_bands_t0.json + bda_composite_prepost_bands_t0_features.json (521 features)
  Saved: bda_composite_prepost_bands_t0.parquet (197114 rows, 586 cols, 16 deltas, 58s)


# CELL 14: A10 -- bda_composite_prepost_landuse (wide)

In [14]:
# @title CELL 14: bda_composite_prepost_landuse_t{tier}.parquet
for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('composite_prepost_landuse', tier)
    if not FR_COMPOSITE_PREPOST_LANDUSE and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  composite_prepost_landuse_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} COMPOSITE PREPOST LANDUSE ---")
    all_city_dfs = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue

        feats = {}
        bids = [f"{city_name}_{i}" for i in range(n_buildings)]

        comp_dir = STACK_ROOT / city_name / "multispectral" / "composites"
        if not comp_dir.exists():
            continue

        for period_dir in sorted(comp_dir.iterdir()):
            if not period_dir.is_dir():
                continue
            period = period_dir.name
            for tif in sorted(period_dir.glob("*landuse*.tif")):
                stem = tif.stem
                col_prefix = f"landuse__{period}"
                data = read_tif_aligned(tif, ref_shape)
                stats = extract_zonal_categorical(label_array, data.astype("int32"), col_prefix, n_buildings, min_pixels=MIN_PIXELS)
                feats.update(stats)

        if not feats:
            continue

        city_df = pd.DataFrame(feats)
        city_df['building_id'] = bids
        city_df['city'] = city_name

        # landuse_changed flag: check if pre vs post dominant class differs
        pre_cols = [c for c in city_df.columns if 'prebattle' in c or 'pre_winter' in c or 'baseline' in c]
        post_cols = [c for c in city_df.columns if 'postbattle' in c or 'post_winter' in c or 'assessment' in c]
        if pre_cols and post_cols:
            pre_mode = city_df[pre_cols[0]] if len(pre_cols) == 1 else city_df[pre_cols].mode(axis=1).iloc[:, 0]
            post_mode = city_df[post_cols[0]] if len(post_cols) == 1 else city_df[post_cols].mode(axis=1).iloc[:, 0]
            city_df['landuse_changed'] = (pre_mode != post_mode).astype(int)

        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        # NaN imputation (Plan Section 6.2)
        df = impute_wide_parquet(df, 'composite_landuse')

        out = save_v2_parquet(df, 'composite_prepost_landuse', tier)
        feat_cols = [c for c in df.columns if c not in (ID_COLS | LABEL_COLS | META_COLS_SET)]
        register_manifest('composite_prepost_landuse', 'A10', 'wide', ['city', 'building_id'], feat_cols,
                         'Q5: Does composite landuse change help?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No composite landuse for tier {tier}")

    gc.collect()



--- Tier 0 COMPOSITE PREPOST LANDUSE ---
  Lysychansk: 30 feature columns
  Mariupol: 30 feature columns
  Rubizhne: 30 feature columns
  Sievierodonetsk: 30 feature columns
    impute_wide_parquet(composite_landuse): 1639554 NaN -> 0 NaN, was_observed_composite_landuse added
    profile -> bda_composite_prepost_landuse_t0.json + bda_composite_prepost_landuse_t0_features.json (29 features)
  Saved: bda_composite_prepost_landuse_t0.parquet (197114 rows, 34 cols, 61s)


# CELL: A11 -- bda_composite_vs_scenes_bands (long, pre-composite vs per-scene post delta)

In [15]:
# @title CELL: bda_composite_vs_scenes_bands_t{tier}.parquet
# For each post-battle MS scene, compute delta from pre-battle composite per band.
# Long-format: 1 row per building x post-battle date.

for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('composite_vs_scenes_bands', tier)
    if not FR_COMPOSITE_VS_SCENES_BANDS and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  composite_vs_scenes_bands_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} COMPOSITE VS SCENES BANDS ---")
    rows = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue
        meta = city_meta[city_name]

        # load pre-battle composite bands
        comp_dir = STACK_ROOT / city_name / "multispectral" / "composites"
        pre_composites = {}
        if comp_dir.exists():
            for period_dir in sorted(comp_dir.iterdir()):
                if not period_dir.is_dir():
                    continue
                pname = period_dir.name
                if 'pre' not in pname.lower() and 'baseline' not in pname.lower():
                    continue
                for tif in sorted(period_dir.glob("*.tif")):
                    if 'landuse' in tif.stem.lower():
                        continue
                    data = read_tif_aligned(tif, ref_shape)
                    stats = extract_zonal_aware(label_array, data, tif.stem, n_buildings, agg="delta", min_pixels=MIN_PIXELS)
                    for k, v in stats.items():
                        if k.endswith('_mean'):
                            pre_composites[tif.stem] = v

        if not pre_composites:
            print(f"  {city_name}: no pre-composite, skip")
            continue

        # per post-battle scene: compute delta from pre-composite
        ms_dir = STACK_ROOT / city_name / "multispectral" / "flat"
        if not ms_dir.exists():
            continue

        dates = set()
        for f in ms_dir.glob("s2__b02__*.tif"):
            m = re.search(r'__(\d{8})\.tif$', f.name)
            if m:
                d = m.group(1)
                period = get_period_label(d, meta['battle_start'], meta['battle_stop'])
                if period == 'postbattle':
                    dates.add(d)

        ts_map = assign_timesteps(sorted(dates), meta['battle_start'])
        bids = [f"{city_name}_{i}" for i in range(n_buildings)]

        for date_str in sorted(dates):
            feats = {}
            for band in MS_BANDS:
                path = ms_dir / f"s2__{band}__{date_str}.tif"
                if not path.exists():
                    continue
                data = read_tif_aligned(path, ref_shape)
                stats = extract_zonal_aware(label_array, data, f"s2__{band}", n_buildings, agg="raw", min_pixels=MIN_PIXELS)
                # scene value
                scene_mean_key = f"s2__{band}_mean"
                if scene_mean_key in stats:
                    feats[f"scene_s2__{band}_mean"] = stats[scene_mean_key]
                    # delta from pre-composite
                    # match pre-composite by band name (exact segment match, not substring)
                    matched = False
                    for pre_stem, pre_vals in pre_composites.items():
                        # pre_stem is like s2__b02__winter_baseline or composite_b02 etc.
                        # split on __ and check if band appears as exact segment
                        segments = pre_stem.replace('composite_', 's2__').split('__')
                        if band in segments:
                            feats[f"delta_s2__{band}_mean"] = stats[scene_mean_key] - pre_vals
                            matched = True
                            break
                    if not matched and f"s2__{band}" in pre_composites:
                        feats[f"delta_s2__{band}_mean"] = stats[scene_mean_key] - pre_composites[f"s2__{band}"]

            if not feats:
                continue

            date_df = pd.DataFrame(feats)
            date_df['building_id'] = bids
            date_df['city'] = city_name
            date_df['date'] = date_str
            date_df['timestep'] = ts_map.get(date_str, 0)
            date_df['period_label'] = 'postbattle'
            rows.append(date_df)

        print(f"  {city_name}: {len(dates)} post-battle dates")

    if rows:
        df = pd.concat(rows, ignore_index=True)
        out = save_v2_parquet(df, 'composite_vs_scenes_bands', tier)
        feat_cols = [c for c in df.columns if c not in (ID_COLS | LABEL_COLS | META_COLS_SET)]
        register_manifest('composite_vs_scenes_bands', 'A11', 'long', ['city', 'building_id', 'date'], feat_cols,
                         'Q3: Does per-scene trajectory beat static snapshot?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No composite vs scenes data for tier {tier}")

    gc.collect()



--- Tier 0 COMPOSITE VS SCENES BANDS ---
  Lysychansk: 3 post-battle dates
  Mariupol: 7 post-battle dates
  Rubizhne: 3 post-battle dates
  Sievierodonetsk: 5 post-battle dates
  No composite vs scenes data for tier 0


# CELL: A12 -- bda_composite_vs_scenes_landuse (long, pre-composite vs per-scene landuse)

In [16]:
# @title CELL: bda_composite_vs_scenes_landuse_t{tier}.parquet
# For each post-battle landuse scene, check if landuse changed from pre-composite.

for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('composite_vs_scenes_landuse', tier)
    if not FR_COMPOSITE_VS_SCENES_LANDUSE and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  composite_vs_scenes_landuse_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} COMPOSITE VS SCENES LANDUSE ---")
    rows = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue
        meta = city_meta[city_name]

        # load pre-composite landuse
        comp_dir = STACK_ROOT / city_name / "multispectral" / "composites"
        pre_lu = None
        if comp_dir.exists():
            for period_dir in sorted(comp_dir.iterdir()):
                if not period_dir.is_dir():
                    continue
                pname = period_dir.name
                if 'pre' not in pname.lower() and 'baseline' not in pname.lower():
                    continue
                for tif in sorted(period_dir.glob("*landuse*.tif")):
                    data = read_tif_aligned(tif, ref_shape)
                    stats = extract_zonal_categorical(label_array, data.astype("int32"), "pre_landuse", n_buildings, min_pixels=MIN_PIXELS)
                    if stats:
                        pre_lu = stats
                        break
                if pre_lu:
                    break

        if not pre_lu:
            print(f"  {city_name}: no pre-composite landuse, skip")
            continue

        # per post-battle landuse scene
        flat_dir = STACK_ROOT / city_name / "landuse" / "flat"
        if not flat_dir.exists():
            continue

        dates = set()
        for tif in flat_dir.glob("s2__landuse__*.tif"):
            m = re.search(r'__(\d{8})\.tif$', tif.name)
            if m:
                d = m.group(1)
                period = get_period_label(d, meta['battle_start'], meta['battle_stop'])
                if period == 'postbattle':
                    dates.add(d)

        ts_map = assign_timesteps(sorted(dates), meta['battle_start'])
        bids = [f"{city_name}_{i}" for i in range(n_buildings)]

        for date_str in sorted(dates):
            tif_path = flat_dir / f"s2__landuse__{date_str}.tif"
            if not tif_path.exists():
                continue
            data = read_tif_aligned(tif_path, ref_shape)
            stats = extract_zonal_categorical(label_array, data.astype("int32"), "post_landuse", n_buildings, min_pixels=MIN_PIXELS)
            if not stats:
                continue

            feats = dict(stats)
            # add pre-composite landuse columns for reference
            feats.update(pre_lu)
            # landuse_changed flag
            pre_key = [k for k in pre_lu.keys() if 'mode' in k or k.endswith('_mean')]
            post_key = [k for k in stats.keys() if 'mode' in k or k.endswith('_mean')]
            if pre_key and post_key:
                feats['landuse_changed'] = (np.array(stats[post_key[0]]) != np.array(pre_lu[pre_key[0]])).astype(int)

            date_df = pd.DataFrame(feats)
            date_df['building_id'] = bids
            date_df['city'] = city_name
            date_df['date'] = date_str
            date_df['timestep'] = ts_map.get(date_str, 0)
            date_df['period_label'] = 'postbattle'
            rows.append(date_df)

        print(f"  {city_name}: {len(dates)} post-battle landuse dates")

    if rows:
        df = pd.concat(rows, ignore_index=True)
        out = save_v2_parquet(df, 'composite_vs_scenes_landuse', tier)
        feat_cols = [c for c in df.columns if c not in (ID_COLS | LABEL_COLS | META_COLS_SET)]
        register_manifest('composite_vs_scenes_landuse', 'A12', 'long', ['city', 'building_id', 'date'], feat_cols,
                         'Q5: Per-scene landuse trajectory?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No composite vs scenes landuse for tier {tier}")

    gc.collect()



--- Tier 0 COMPOSITE VS SCENES LANDUSE ---
  Lysychansk: 6 post-battle landuse dates
  Mariupol: 7 post-battle landuse dates
  Rubizhne: 6 post-battle landuse dates
  Sievierodonetsk: 8 post-battle landuse dates
    profile -> bda_composite_vs_scenes_landuse_t0.json + bda_composite_vs_scenes_landuse_t0_features.json (19 features)
  Saved: bda_composite_vs_scenes_landuse_t0.parquet (1330093 rows, 26 cols, 6s)


# CELL 15: A13 -- bda_prepost_single_card (wide, single-scene CARD pre/post)

In [17]:
# @title CELL 15: bda_prepost_single_card_t{tier}.parquet
# v29: legacy helper kept for backward reference. Use extract_zonal_aware below.
def extract_building_stats_vectorized(label_array, data_2d, n_buildings, min_px=MIN_PIXELS):
    """DEPRECATED: returns mean+std only. Use extract_zonal_aware(...) instead."""
    means = np.full(n_buildings, np.nan, dtype=np.float32)
    stds = np.full(n_buildings, np.nan, dtype=np.float32)

    flat_labels = label_array.ravel()
    flat_data = data_2d.ravel()

    valid = np.isfinite(flat_data) & (flat_labels > 0) & (flat_labels <= n_buildings)
    flat_labels_v = flat_labels[valid]
    flat_data_v = flat_data[valid]

    if len(flat_data_v) == 0:
        return means, stds

    order = np.argsort(flat_labels_v)
    sorted_labels = flat_labels_v[order]
    sorted_data = flat_data_v[order]

    unique_labels, start_idx, counts = np.unique(sorted_labels, return_index=True, return_counts=True)

    for ul, si, cnt in zip(unique_labels, start_idx, counts):
        bid = int(ul) - 1
        if bid < 0 or bid >= n_buildings:
            continue
        if cnt < min_px:
            continue
        vals = sorted_data[si:si + cnt]
        means[bid] = np.mean(vals)
        stds[bid] = np.std(vals)

    return means, stds

for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('prepost_single_card', tier)
    if not FR_PREPOST_SINGLE_CARD and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  prepost_single_card_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} PREPOST SINGLE CARD ---")
    all_city_dfs = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue

        bids = [f"{city_name}_{i}" for i in range(n_buildings)]
        feats = {}

        card_dir = STACK_ROOT / city_name / "SAR_CARD" / SAR_CARD_PREPOST_SUBDIR
        card_meta_path = card_dir / "card_prepost_meta.json"
        pre_date_card = None
        post_date_card = None

        if card_dir.exists():
            if card_meta_path.exists():
                with open(card_meta_path) as f:
                    card_meta = json.load(f)
                pre_date_card = card_meta.get('vv_pre_date')
                post_date_card = card_meta.get('vv_post_date')

            for pol in ['vv', 'vh']:
                for phase in ['pre', 'post', 'delta']:
                    if phase != 'delta':
                        pattern = f"s1__{pol}__card_prepost__{phase}__*.tif"
                    else:
                        pattern = f"s1__{pol}__card_prepost__delta.tif"
                    matches = list(card_dir.glob(pattern))
                    if not matches:
                        continue
                    tif_path = matches[0]
                    data = read_tif_aligned(tif_path, ref_shape)
                    # v29: full reduction set per phase. agg='delta' for delta phase, 'raw' for pre/post.
                    prefix = f"{phase}_{pol}"
                    agg = 'delta' if phase == 'delta' else 'raw'
                    stats = extract_zonal_aware(label_array, data, prefix, n_buildings,
                                                  agg=agg, min_pixels=MIN_PIXELS)
                    feats.update(stats)

        if not feats:
            continue

        city_df = pd.DataFrame(feats)
        city_df['building_id'] = bids
        city_df['city'] = city_name
        if pre_date_card:
            city_df['pre_date_card'] = pre_date_card
        if post_date_card:
            city_df['post_date_card'] = post_date_card

        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feats, card={pre_date_card}->{post_date_card}")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        # merge damage labels from buildings parquet (v1 compatibility for NB08b xBD comparison)
        bldg_path = v2_path('buildings', tier)
        if bldg_path.exists():
            df_bldg = pd.read_parquet(bldg_path, columns=['building_id', 'city', 'damage_binary'])
            df = df.merge(df_bldg, on=['building_id', 'city'], how='left')
            df.rename(columns={'damage_binary': 'damage_label'}, inplace=True)
        else:
            print(f"  WARNING: buildings parquet not found for tier {tier}")

        # NaN imputation (Plan Section 6.2)
        df = impute_wide_parquet(df, 'card_prepost')

        out = save_v2_parquet(df, 'prepost_single_card', tier)
        feat_cols = [c for c in df.columns if c not in (ID_COLS | LABEL_COLS | META_COLS_SET)]
        register_manifest('prepost_single_card', 'A13', 'wide', ['city', 'building_id'], feat_cols,
                         'Q3: Single-scene CARD pre/post', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No single CARD prepost for tier {tier}")

    gc.collect()



--- Tier 0 PREPOST SINGLE CARD ---
  Lysychansk: 54 feats, card=20220608->20220726
  Mariupol: 54 feats, card=20220204->20220604
  Rubizhne: 54 feats, card=20220211->20220530
  Sievierodonetsk: 54 feats, card=20220412->20220717
    impute_wide_parquet(card_prepost): 2478420 NaN -> 0 NaN, was_observed_card_prepost added
    profile -> bda_prepost_single_card_t0.json + bda_prepost_single_card_t0_features.json (49 features)
  Saved: bda_prepost_single_card_t0.parquet (197114 rows, 60 cols, 8s)


# CELL 16: A14 -- bda_coh_drop (wide, accumulator)

In [18]:
# @title CELL 16: bda_coh_drop_t{tier}.parquet
# v29 per-TIF dispatch helper for accumulator cells
def _dispatch_accumulator_tif(label_array, data, prefix, n_buildings):
    """Dispatch the right helper based on TIF stem keywords.

    Returns dict of column -> array. Handles:
      running_min                                    -> agg='accum_min'
      running_max / max_*drop / max_abs_delta / nbr_z_abs_running_max -> agg='accum_max'
      *_count / *_rate / scenes_observed / urban_retained / loss_fraction -> agg='count'
      date_first_*, date_worst_*, date_persistent_*  -> extract_zonal_dates (METADATA)
      lu_* / *_class                                  -> extract_zonal_categorical
      *baseline* / *fourier* / *winter*              -> agg='raw' (pre-event statistics)
    """
    p = prefix.lower()
    if 'date_first' in p or 'date_worst' in p or 'date_persistent' in p or 'date_' in p.split('__')[-1]:
        return extract_zonal_dates(label_array, data, prefix, n_buildings, min_pixels=MIN_PIXELS)
    if 'lu_transition' in p or 'lu_at_' in p or 'modal_post_class' in p or 'final_class' in p:
        return extract_zonal_categorical(label_array, data.astype('int32'), prefix, n_buildings, min_pixels=MIN_PIXELS)
    if 'running_min' in p or '__min' == p[-5:]:
        return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='accum_min', min_pixels=MIN_PIXELS)
    if 'running_max' in p or 'max_drop' in p or 'max_z_drop' in p or 'max_abs_delta' in p or 'nbr_z_abs_running_max' in p or 'swir_z_running_max' in p or 'mahalanobis_running_max' in p:
        return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='accum_max', min_pixels=MIN_PIXELS)
    if 'baseline' in p or 'fourier' in p or 'winter' in p:
        return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='raw', min_pixels=MIN_PIXELS)
    if '_count' in p or '_rate' in p or 'scenes_observed' in p or 'urban_retained' in p or 'loss_fraction' in p or 'exceedance' in p or 'rise_count' in p or 'drop_count' in p or 'anomaly_count' in p:
        return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='count', min_pixels=MIN_PIXELS)
    return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='raw', min_pixels=MIN_PIXELS)

COH_DROP_PRODUCTS = [
    's1__coh__running_min.tif',
    's1__coh__drop_count.tif',
    's1__coh__date_first_drop.tif',
    's1__coh__date_worst_drop.tif',
    's1__coh__max_drop.tif',
    's1__coh__scenes_observed.tif',
    's1__coh__lu_transition.tif',
]

for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('coh_drop', tier)
    if not FR_COH_DROP and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  coh_drop_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} COH Drop Accumulator ---")
    all_city_dfs = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue

        drop_dir = STACK_ROOT / city_name / "temporal" / "COH" / "coh_drop_accumulator"
        if not drop_dir.exists():
            print(f"  {city_name}: no coh_drop_accumulator, skipping")
            continue

        feats = {}
        bids = [f"{city_name}_{i}" for i in range(n_buildings)]

        for tif_name in COH_DROP_PRODUCTS:
            tif_path = drop_dir / tif_name
            if not tif_path.exists():
                continue

            prefix = tif_path.stem
            data = read_tif_aligned(tif_path, ref_shape)
            # v29: dispatch per TIF stem
            stats = _dispatch_accumulator_tif(label_array, data, prefix, n_buildings)
            feats.update(stats)

        if not feats:
            print(f"  {city_name}: no features extracted")
            continue

        city_df = pd.DataFrame(feats)
        city_df['building_id'] = bids
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns, {n_buildings} buildings")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        meta_cols = ['building_id', 'city']
        feat_cols = sorted([c for c in df.columns if c not in meta_cols])
        df = df[meta_cols + feat_cols]

        # NaN imputation (Plan Section 6.2) -- was_observed_cohdrop set per-row by helper
        # cities with no cohdrop TIFs are intentionally NOT expanded to all buildings
        # (prevents was_observed=cohdrop=0 from becoming a city-identifier / GroupKFold leakage)
        df = impute_wide_parquet(df, 'cohdrop')

        out = save_v2_parquet(df, 'coh_drop', tier)
        register_manifest('coh_drop', 'A14', 'wide', ['city', 'building_id'], feat_cols,
                         'Q1: Does cumulative COH drop carry signal?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No COH drop data for tier {tier}")

    gc.collect()



--- Tier 0 COH Drop Accumulator ---
  Lysychansk: 44 feature columns, 47169 buildings
  Mariupol: 44 feature columns, 119107 buildings
  Rubizhne: no coh_drop_accumulator, skipping
  Sievierodonetsk: 44 feature columns, 14151 buildings
    impute_wide_parquet(cohdrop): 4470226 NaN -> 0 NaN, was_observed_cohdrop added
    profile -> bda_coh_drop_t0.json + bda_coh_drop_t0_features.json (31 features)
  Saved: bda_coh_drop_t0.parquet (180427 rows, 47 cols, 2s)


# CELL 16b: A19 -- bda_card_drop (wide, accumulator)
Zonal stats of NB03e R4 CARD drop accumulator TIFs per building. Categorical
`lu_transition` handled via `is_landuse=True`. Missing-per-city rows get NaN then
median imputation plus `was_observed_carddrop` flag.

In [19]:
# @title CELL 16b: bda_card_drop_t{tier}.parquet
# v29 per-TIF dispatch helper for accumulator cells
def _dispatch_accumulator_tif(label_array, data, prefix, n_buildings):
    """Dispatch the right helper based on TIF stem keywords.

    Returns dict of column -> array. Handles:
      running_min                                    -> agg='accum_min'
      running_max / max_*drop / max_abs_delta / nbr_z_abs_running_max -> agg='accum_max'
      *_count / *_rate / scenes_observed / urban_retained / loss_fraction -> agg='count'
      date_first_*, date_worst_*, date_persistent_*  -> extract_zonal_dates (METADATA)
      lu_* / *_class                                  -> extract_zonal_categorical
      *baseline* / *fourier* / *winter*              -> agg='raw' (pre-event statistics)
    """
    p = prefix.lower()
    if 'date_first' in p or 'date_worst' in p or 'date_persistent' in p or 'date_' in p.split('__')[-1]:
        return extract_zonal_dates(label_array, data, prefix, n_buildings, min_pixels=MIN_PIXELS)
    if 'lu_transition' in p or 'lu_at_' in p or 'modal_post_class' in p or 'final_class' in p:
        return extract_zonal_categorical(label_array, data.astype('int32'), prefix, n_buildings, min_pixels=MIN_PIXELS)
    if 'running_min' in p or '__min' == p[-5:]:
        return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='accum_min', min_pixels=MIN_PIXELS)
    if 'running_max' in p or 'max_drop' in p or 'max_z_drop' in p or 'max_abs_delta' in p or 'nbr_z_abs_running_max' in p or 'swir_z_running_max' in p or 'mahalanobis_running_max' in p:
        return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='accum_max', min_pixels=MIN_PIXELS)
    if 'baseline' in p or 'fourier' in p or 'winter' in p:
        return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='raw', min_pixels=MIN_PIXELS)
    if '_count' in p or '_rate' in p or 'scenes_observed' in p or 'urban_retained' in p or 'loss_fraction' in p or 'exceedance' in p or 'rise_count' in p or 'drop_count' in p or 'anomaly_count' in p:
        return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='count', min_pixels=MIN_PIXELS)
    return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='raw', min_pixels=MIN_PIXELS)

CARD_DROP_PRODUCTS = [
    's1__vv__z_running_min.tif',
    's1__vv__drop_count.tif',
    's1__vv__date_first_drop.tif',
    's1__vv__date_worst_drop.tif',
    's1__vv__max_z_drop.tif',
    's1__vv__scenes_observed.tif',
    's1__vv__lu_transition.tif',
]

for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('card_drop', tier)
    if not FR_CARD_DROP and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  card_drop_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} CARD Drop Accumulator ---")
    all_city_dfs = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue

        drop_dir = STACK_ROOT / city_name / "temporal" / "CARD" / "card_drop_accumulator"
        if not drop_dir.exists():
            print(f"  {city_name}: no card_drop_accumulator, skipping")
            continue

        feats = {}
        bids = [f"{city_name}_{i}" for i in range(n_buildings)]

        for tif_name in CARD_DROP_PRODUCTS:
            tif_path = drop_dir / tif_name
            if not tif_path.exists():
                continue

            prefix = tif_path.stem
            data = read_tif_aligned(tif_path, ref_shape)
            # v29: dispatch per TIF stem
            stats = _dispatch_accumulator_tif(label_array, data, prefix, n_buildings)
            feats.update(stats)

        if not feats:
            print(f"  {city_name}: no features extracted")
            continue

        city_df = pd.DataFrame(feats)
        city_df['building_id'] = bids
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns, {n_buildings} buildings")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        meta_cols = ['building_id', 'city']
        feat_cols = sorted([c for c in df.columns if c not in meta_cols])
        df = df[meta_cols + feat_cols]

        # NaN imputation (Plan Section 6.2) -- was_observed_carddrop set per-row by helper
        # cities with no carddrop TIFs are intentionally NOT expanded to all buildings
        # (prevents was_observed=carddrop=0 from becoming a city-identifier / GroupKFold leakage)
        df = impute_wide_parquet(df, 'carddrop')

        out = save_v2_parquet(df, 'card_drop', tier)
        register_manifest('card_drop', 'A19', 'wide', ['city', 'building_id'], feat_cols,
                         'Q: Does cumulative CARD z-score drop carry signal?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No CARD drop data for tier {tier}")

    gc.collect()



--- Tier 0 CARD Drop Accumulator ---
  Lysychansk: 44 feature columns, 47169 buildings
  Mariupol: 44 feature columns, 119107 buildings
  Rubizhne: 44 feature columns, 16687 buildings
  Sievierodonetsk: 44 feature columns, 14151 buildings
    impute_wide_parquet(carddrop): 4284277 NaN -> 0 NaN, was_observed_carddrop added
    profile -> bda_card_drop_t0.json + bda_card_drop_t0_features.json (32 features)
  Saved: bda_card_drop_t0.parquet (197114 rows, 47 cols, 1s)


# CELL 16c: A20 -- bda_ms_change (wide, accumulator)
Zonal stats of NB03e R5 MS change accumulator TIFs per building (SWIR brightness
running-max z, NBR anomaly magnitude, rise/anomaly counts, first/worst dates,
baseline mean/std per band, lu_transition). Categorical `lu_transition` handled
via `is_landuse=True`.

In [20]:
# @title CELL 16c: bda_ms_change_t{tier}.parquet
# v29 per-TIF dispatch helper for accumulator cells
def _dispatch_accumulator_tif(label_array, data, prefix, n_buildings):
    """Dispatch the right helper based on TIF stem keywords.

    Returns dict of column -> array. Handles:
      running_min                                    -> agg='accum_min'
      running_max / max_*drop / max_abs_delta / nbr_z_abs_running_max -> agg='accum_max'
      *_count / *_rate / scenes_observed / urban_retained / loss_fraction -> agg='count'
      date_first_*, date_worst_*, date_persistent_*  -> extract_zonal_dates (METADATA)
      lu_* / *_class                                  -> extract_zonal_categorical
      *baseline* / *fourier* / *winter*              -> agg='raw' (pre-event statistics)
    """
    p = prefix.lower()
    if 'date_first' in p or 'date_worst' in p or 'date_persistent' in p or 'date_' in p.split('__')[-1]:
        return extract_zonal_dates(label_array, data, prefix, n_buildings, min_pixels=MIN_PIXELS)
    if 'lu_transition' in p or 'lu_at_' in p or 'modal_post_class' in p or 'final_class' in p:
        return extract_zonal_categorical(label_array, data.astype('int32'), prefix, n_buildings, min_pixels=MIN_PIXELS)
    if 'running_min' in p or '__min' == p[-5:]:
        return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='accum_min', min_pixels=MIN_PIXELS)
    if 'running_max' in p or 'max_drop' in p or 'max_z_drop' in p or 'max_abs_delta' in p or 'nbr_z_abs_running_max' in p or 'swir_z_running_max' in p or 'mahalanobis_running_max' in p:
        return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='accum_max', min_pixels=MIN_PIXELS)
    if 'baseline' in p or 'fourier' in p or 'winter' in p:
        return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='raw', min_pixels=MIN_PIXELS)
    if '_count' in p or '_rate' in p or 'scenes_observed' in p or 'urban_retained' in p or 'loss_fraction' in p or 'exceedance' in p or 'rise_count' in p or 'drop_count' in p or 'anomaly_count' in p:
        return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='count', min_pixels=MIN_PIXELS)
    return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='raw', min_pixels=MIN_PIXELS)

MS_CHANGE_PRODUCTS = [
    's2__swir_z_running_max.tif',
    's2__nbr_z_abs_running_max.tif',
    's2__swir_rise_count.tif',
    's2__nbr_anomaly_count.tif',
    's2__date_first_swir_rise.tif',
    's2__date_worst_swir_rise.tif',
    's2__scenes_observed.tif',
    's2__lu_transition.tif',
    's2__swir_baseline_mean.tif',
    's2__swir_baseline_std.tif',
    's2__nbr_baseline_mean.tif',
    's2__nbr_baseline_std.tif',
]

for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('ms_change', tier)
    if not FR_MS_CHANGE and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  ms_change_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} MS Change Accumulator ---")
    all_city_dfs = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue

        acc_dir = STACK_ROOT / city_name / "temporal" / "MS" / "ms_change_accumulator"
        if not acc_dir.exists():
            print(f"  {city_name}: no ms_change_accumulator, skipping")
            continue

        feats = {}
        bids = [f"{city_name}_{i}" for i in range(n_buildings)]

        for tif_name in MS_CHANGE_PRODUCTS:
            tif_path = acc_dir / tif_name
            if not tif_path.exists():
                continue

            prefix = tif_path.stem
            data = read_tif_aligned(tif_path, ref_shape)
            # v29: dispatch per TIF stem
            stats = _dispatch_accumulator_tif(label_array, data, prefix, n_buildings)
            feats.update(stats)

        if not feats:
            print(f"  {city_name}: no features extracted")
            continue

        city_df = pd.DataFrame(feats)
        city_df['building_id'] = bids
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns, {n_buildings} buildings")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        meta_cols = ['building_id', 'city']
        feat_cols = sorted([c for c in df.columns if c not in meta_cols])
        df = df[meta_cols + feat_cols]

        # NaN imputation (Plan Section 6.2) -- was_observed_mschange set per-row by helper
        # cities with no mschange TIFs are intentionally NOT expanded to all buildings
        # (prevents was_observed=mschange=0 from becoming a city-identifier / GroupKFold leakage)
        df = impute_wide_parquet(df, 'mschange')

        out = save_v2_parquet(df, 'ms_change', tier)
        register_manifest('ms_change', 'A20', 'wide', ['city', 'building_id'], feat_cols,
                         'Q: Does cumulative MS SWIR brightness + NBR anomaly carry signal?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No MS change data for tier {tier}")

    gc.collect()



--- Tier 0 MS Change Accumulator ---
  Lysychansk: 48 feature columns, 47169 buildings
  Mariupol: no ms_change_accumulator, skipping
  Rubizhne: 30 feature columns, 16687 buildings
  Sievierodonetsk: 30 feature columns, 14151 buildings
    impute_wide_parquet(mschange): 2496224 NaN -> 2496224 NaN, was_observed_mschange added
    profile -> bda_ms_change_t0.json + bda_ms_change_t0_features.json (35 features)
  Saved: bda_ms_change_t0.parquet (78007 rows, 51 cols, 1s)


# CELL 16d: A21 -- bda_ms_maha (wide, accumulator)
Zonal stats of NB03e R7 MS Mahalanobis distance accumulator TIFs per building
(running-max Mahalanobis distance across all MS bands, exceedance count,
first/worst exceedance dates, scenes observed, lu_transition).

In [21]:
# @title CELL 16d: bda_ms_maha_t{tier}.parquet
# v29 per-TIF dispatch helper for accumulator cells
def _dispatch_accumulator_tif(label_array, data, prefix, n_buildings):
    """Dispatch the right helper based on TIF stem keywords.

    Returns dict of column -> array. Handles:
      running_min                                    -> agg='accum_min'
      running_max / max_*drop / max_abs_delta / nbr_z_abs_running_max -> agg='accum_max'
      *_count / *_rate / scenes_observed / urban_retained / loss_fraction -> agg='count'
      date_first_*, date_worst_*, date_persistent_*  -> extract_zonal_dates (METADATA)
      lu_* / *_class                                  -> extract_zonal_categorical
      *baseline* / *fourier* / *winter*              -> agg='raw' (pre-event statistics)
    """
    p = prefix.lower()
    if 'date_first' in p or 'date_worst' in p or 'date_persistent' in p or 'date_' in p.split('__')[-1]:
        return extract_zonal_dates(label_array, data, prefix, n_buildings, min_pixels=MIN_PIXELS)
    if 'lu_transition' in p or 'lu_at_' in p or 'modal_post_class' in p or 'final_class' in p:
        return extract_zonal_categorical(label_array, data.astype('int32'), prefix, n_buildings, min_pixels=MIN_PIXELS)
    if 'running_min' in p or '__min' == p[-5:]:
        return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='accum_min', min_pixels=MIN_PIXELS)
    if 'running_max' in p or 'max_drop' in p or 'max_z_drop' in p or 'max_abs_delta' in p or 'nbr_z_abs_running_max' in p or 'swir_z_running_max' in p or 'mahalanobis_running_max' in p:
        return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='accum_max', min_pixels=MIN_PIXELS)
    if 'baseline' in p or 'fourier' in p or 'winter' in p:
        return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='raw', min_pixels=MIN_PIXELS)
    if '_count' in p or '_rate' in p or 'scenes_observed' in p or 'urban_retained' in p or 'loss_fraction' in p or 'exceedance' in p or 'rise_count' in p or 'drop_count' in p or 'anomaly_count' in p:
        return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='count', min_pixels=MIN_PIXELS)
    return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='raw', min_pixels=MIN_PIXELS)

MS_MAHA_PRODUCTS = [
    's2__mahalanobis_running_max.tif',
    's2__mahalanobis_exceedance_count.tif',
    's2__date_first_exceedance.tif',
    's2__date_worst_exceedance.tif',
    's2__scenes_observed.tif',
    's2__lu_transition.tif',
]

for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('ms_maha', tier)
    if not FR_MS_MAHA and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  ms_maha_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} MS Mahalanobis Accumulator ---")
    all_city_dfs = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue

        acc_dir = STACK_ROOT / city_name / "temporal" / "MS" / "ms_mahalanobis_accumulator"
        if not acc_dir.exists():
            print(f"  {city_name}: no ms_mahalanobis_accumulator, skipping")
            continue

        feats = {}
        bids = [f"{city_name}_{i}" for i in range(n_buildings)]

        for tif_name in MS_MAHA_PRODUCTS:
            tif_path = acc_dir / tif_name
            if not tif_path.exists():
                continue

            prefix = tif_path.stem
            data = read_tif_aligned(tif_path, ref_shape)
            # v29: dispatch per TIF stem
            stats = _dispatch_accumulator_tif(label_array, data, prefix, n_buildings)
            feats.update(stats)

        if not feats:
            print(f"  {city_name}: no features extracted")
            continue

        city_df = pd.DataFrame(feats)
        city_df['building_id'] = bids
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns, {n_buildings} buildings")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        meta_cols = ['building_id', 'city']
        feat_cols = sorted([c for c in df.columns if c not in meta_cols])
        df = df[meta_cols + feat_cols]

        # NaN imputation (Plan Section 6.2) -- was_observed_msmaha set per-row by helper
        # cities with no msmaha TIFs are intentionally NOT expanded to all buildings
        # (prevents was_observed=msmaha=0 from becoming a city-identifier / GroupKFold leakage)
        df = impute_wide_parquet(df, 'msmaha')

        out = save_v2_parquet(df, 'ms_maha', tier)
        register_manifest('ms_maha', 'A21', 'wide', ['city', 'building_id'], feat_cols,
                         'Q: Does multi-band MS Mahalanobis distance carry signal?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No MS Mahalanobis data for tier {tier}")

    gc.collect()



--- Tier 0 MS Mahalanobis Accumulator ---
  Lysychansk: 35 feature columns, 47169 buildings
  Mariupol: no ms_mahalanobis_accumulator, skipping
  Rubizhne: 35 feature columns, 16687 buildings
  Sievierodonetsk: 35 feature columns, 14151 buildings
    impute_wide_parquet(msmaha): 875221 NaN -> 0 NaN, was_observed_msmaha added
    profile -> bda_ms_maha_t0.json + bda_ms_maha_t0_features.json (24 features)
  Saved: bda_ms_maha_t0.parquet (78007 rows, 38 cols, 1s)


# CELL 16e: A22 -- bda_lu_change (wide, accumulator)
Zonal stats of NB03e R6 Landuse change accumulator TIFs per building (date of
first urban->other loss, date of persistent loss, loss count/fraction,
final/modal-post class, urban retained flag, scenes observed). Categorical
`final_class`, `modal_post_class` handled via `is_landuse=True`; `urban_retained`
is kept numeric (fraction retained within building footprint).

In [22]:
# @title CELL 16e: bda_lu_change_t{tier}.parquet
# v29 per-TIF dispatch helper for accumulator cells
def _dispatch_accumulator_tif(label_array, data, prefix, n_buildings):
    """Dispatch the right helper based on TIF stem keywords.

    Returns dict of column -> array. Handles:
      running_min                                    -> agg='accum_min'
      running_max / max_*drop / max_abs_delta / nbr_z_abs_running_max -> agg='accum_max'
      *_count / *_rate / scenes_observed / urban_retained / loss_fraction -> agg='count'
      date_first_*, date_worst_*, date_persistent_*  -> extract_zonal_dates (METADATA)
      lu_* / *_class                                  -> extract_zonal_categorical
      *baseline* / *fourier* / *winter*              -> agg='raw' (pre-event statistics)
    """
    p = prefix.lower()
    if 'date_first' in p or 'date_worst' in p or 'date_persistent' in p or 'date_' in p.split('__')[-1]:
        return extract_zonal_dates(label_array, data, prefix, n_buildings, min_pixels=MIN_PIXELS)
    if 'lu_transition' in p or 'lu_at_' in p or 'modal_post_class' in p or 'final_class' in p:
        return extract_zonal_categorical(label_array, data.astype('int32'), prefix, n_buildings, min_pixels=MIN_PIXELS)
    if 'running_min' in p or '__min' == p[-5:]:
        return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='accum_min', min_pixels=MIN_PIXELS)
    if 'running_max' in p or 'max_drop' in p or 'max_z_drop' in p or 'max_abs_delta' in p or 'nbr_z_abs_running_max' in p or 'swir_z_running_max' in p or 'mahalanobis_running_max' in p:
        return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='accum_max', min_pixels=MIN_PIXELS)
    if 'baseline' in p or 'fourier' in p or 'winter' in p:
        return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='raw', min_pixels=MIN_PIXELS)
    if '_count' in p or '_rate' in p or 'scenes_observed' in p or 'urban_retained' in p or 'loss_fraction' in p or 'exceedance' in p or 'rise_count' in p or 'drop_count' in p or 'anomaly_count' in p:
        return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='count', min_pixels=MIN_PIXELS)
    return extract_zonal_aware(label_array, data, prefix, n_buildings, agg='raw', min_pixels=MIN_PIXELS)

LU_CHANGE_PRODUCTS = [
    's2__lu__date_first_loss.tif',
    's2__lu__date_persistent_loss.tif',
    's2__lu__loss_count.tif',
    's2__lu__loss_fraction.tif',
    's2__lu__final_class.tif',
    's2__lu__modal_post_class.tif',
    's2__lu__urban_retained.tif',
    's2__lu__scenes_observed.tif',
]

for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('lu_change', tier)
    if not FR_LU_CHANGE and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  lu_change_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} Landuse Change Accumulator ---")
    all_city_dfs = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue

        acc_dir = STACK_ROOT / city_name / "temporal" / "LANDUSE" / "landuse_change_accumulator"
        if not acc_dir.exists():
            print(f"  {city_name}: no landuse_change_accumulator, skipping")
            continue

        feats = {}
        bids = [f"{city_name}_{i}" for i in range(n_buildings)]

        for tif_name in LU_CHANGE_PRODUCTS:
            tif_path = acc_dir / tif_name
            if not tif_path.exists():
                continue

            prefix = tif_path.stem
            data = read_tif_aligned(tif_path, ref_shape)
            # v29: dispatch per TIF stem
            stats = _dispatch_accumulator_tif(label_array, data, prefix, n_buildings)
            feats.update(stats)

        if not feats:
            print(f"  {city_name}: no features extracted")
            continue

        city_df = pd.DataFrame(feats)
        city_df['building_id'] = bids
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns, {n_buildings} buildings")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        meta_cols = ['building_id', 'city']
        feat_cols = sorted([c for c in df.columns if c not in meta_cols])
        df = df[meta_cols + feat_cols]

        # NaN imputation (Plan Section 6.2) -- was_observed_luchange set per-row by helper
        # cities with no luchange TIFs are intentionally NOT expanded to all buildings
        # (prevents was_observed=luchange=0 from becoming a city-identifier / GroupKFold leakage)
        df = impute_wide_parquet(df, 'luchange')

        out = save_v2_parquet(df, 'lu_change', tier)
        register_manifest('lu_change', 'A22', 'wide', ['city', 'building_id'], feat_cols,
                         'Q: Does persistent urban-to-other landuse loss carry signal?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No Landuse change data for tier {tier}")

    gc.collect()



--- Tier 0 Landuse Change Accumulator ---
  Lysychansk: 44 feature columns, 47169 buildings
  Mariupol: 44 feature columns, 119107 buildings
  Rubizhne: 44 feature columns, 16687 buildings
  Sievierodonetsk: 44 feature columns, 14151 buildings
    impute_wide_parquet(luchange): 3488481 NaN -> 0 NaN, was_observed_luchange added
    profile -> bda_lu_change_t0.json + bda_lu_change_t0_features.json (31 features)
  Saved: bda_lu_change_t0.parquet (197114 rows, 47 cols, 3s)


# CELL 17: A15 -- bda_block_stats (wide, Dietrich replication)

In [23]:
# @title CELL 17: bda_block_stats_t{tier}.parquet
for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('block_stats', tier)
    if not FR_BLOCK_STATS and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  block_stats_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} BLOCK STATS ---")
    all_city_dfs = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue

        feats = {}
        bids = [f"{city_name}_{i}" for i in range(n_buildings)]

        # CARD baseline
        bl_card_dir = STACK_ROOT / city_name / "SAR_CARD" / "temporal_stats"
        if bl_card_dir.exists():
            for tif in sorted(bl_card_dir.glob("s1__*__baseline__*.tif")):
                data = read_tif_aligned(tif, ref_shape)
                stats = extract_zonal_aware(label_array, data, tif.stem, n_buildings, agg="stat_of_stat", min_pixels=MIN_PIXELS)
                feats.update(stats)

        # COH baseline
        bl_coh_dir = STACK_ROOT / city_name / "SAR_SLC" / "coherence_baseline"
        if bl_coh_dir.exists():
            for tif in sorted(bl_coh_dir.glob("*.tif")):
                data = read_tif_aligned(tif, ref_shape)
                stats = extract_zonal_aware(label_array, data, tif.stem, n_buildings, agg="stat_of_stat", min_pixels=MIN_PIXELS)
                feats.update(stats)

        # COH post-battle baseline
        coh_post_dir = STACK_ROOT / city_name / "temporal" / "COH" / "post_baseline"
        if coh_post_dir.exists():
            for tif in sorted(coh_post_dir.glob("*.tif")):
                data = read_tif_aligned(tif, ref_shape)
                stats = extract_zonal_aware(label_array, data, tif.stem, n_buildings, agg="stat_of_stat", min_pixels=MIN_PIXELS)
                feats.update(stats)

        # Block stats
        bs_dir = STACK_ROOT / city_name / "temporal" / "CARD" / "block_stats"
        if bs_dir.exists():
            for tif in sorted(bs_dir.glob("s1__*.tif")):
                data = read_tif_aligned(tif, ref_shape)
                stats = extract_zonal_aware(label_array, data, tif.stem, n_buildings, agg="stat_of_stat", min_pixels=MIN_PIXELS)
                feats.update(stats)

        if not feats:
            continue

        city_df = pd.DataFrame(feats)
        city_df['building_id'] = bids
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        # NaN imputation (Plan Section 6.2)
        df = impute_wide_parquet(df, 'blockstats')

        out = save_v2_parquet(df, 'block_stats', tier)
        feat_cols = [c for c in df.columns if c not in (ID_COLS | LABEL_COLS | META_COLS_SET)]
        register_manifest('block_stats', 'A15', 'wide', ['city', 'building_id'], feat_cols,
                         'Q6: Does Dietrich block approach work with GroupKFold?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No block stats for tier {tier}")

    gc.collect()



--- Tier 0 BLOCK STATS ---
  Lysychansk: 560 feature columns
  Mariupol: 742 feature columns
  Rubizhne: 518 feature columns
  Sievierodonetsk: 518 feature columns
    impute_wide_parquet(blockstats): 65313607 NaN -> 23653680 NaN, was_observed_blockstats added
    profile -> bda_block_stats_t0.json + bda_block_stats_t0_features.json (636 features)
  Saved: bda_block_stats_t0.parquet (197114 rows, 745 cols, 71s)


# CELL 18: A16 -- bda_rolling_stats_roll3 (wide)

In [24]:
# @title CELL 18: bda_rolling_stats_roll3_t{tier}.parquet
def _extract_dir(label_array, ref_shape, n_buildings, tif_dir, prefix_filter=None):
    feats = {}
    if not tif_dir.exists():
        return feats, 0
    n = 0
    for tif in sorted(tif_dir.glob("*.tif")):
        stem = tif.stem
        if prefix_filter and not any(stem.startswith(p) for p in prefix_filter):
            continue
        data = read_tif_aligned(tif, ref_shape)
        stats = extract_zonal_aware(label_array, data, stem, n_buildings, agg="stat_of_stat", min_pixels=MIN_PIXELS)
        feats.update(stats)
        n += 1
    return feats, n

for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('rolling_stats_roll3', tier)
    if not FR_ROLLING_STATS and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  rolling_stats_roll3_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} ROLLING STATS ROLL3 ---")
    all_city_dfs = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue

        feats = {}
        bids = [f"{city_name}_{i}" for i in range(n_buildings)]

        bl_card_dir = STACK_ROOT / city_name / "SAR_CARD" / "temporal_stats"
        f1, n1 = _extract_dir(label_array, ref_shape, n_buildings, bl_card_dir,
                               prefix_filter=["s1__vv__baseline", "s1__vh__baseline"])
        feats.update(f1)

        bl_coh_dir = STACK_ROOT / city_name / "SAR_SLC" / "coherence_baseline"
        f2, n2 = _extract_dir(label_array, ref_shape, n_buildings, bl_coh_dir)
        feats.update(f2)

        rs_card_dir = STACK_ROOT / city_name / "temporal" / "CARD" / "rolling_stats"
        f3, n3 = _extract_dir(label_array, ref_shape, n_buildings, rs_card_dir,
                               prefix_filter=[f"s1__vv__roll3__", f"s1__vh__roll3__"])
        feats.update(f3)

        rs_coh_dir = STACK_ROOT / city_name / "temporal" / "COH" / "rolling_stats"
        f4, n4 = _extract_dir(label_array, ref_shape, n_buildings, rs_coh_dir,
                               prefix_filter=[f"s1__coh_vv__roll3__"])
        feats.update(f4)

        if not feats:
            continue

        city_df = pd.DataFrame(feats)
        city_df['building_id'] = bids
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        # NaN imputation (Plan Section 6.2)
        df = impute_wide_parquet(df, 'rolling_stats')

        out = save_v2_parquet(df, 'rolling_stats_roll3', tier)
        feat_cols = [c for c in df.columns if c not in (ID_COLS | LABEL_COLS | META_COLS_SET)]
        register_manifest('rolling_stats_roll3', 'A16', 'wide', ['city', 'building_id'], feat_cols,
                         'Q4: Rolling stats (window=3)', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No rolling stats roll3 for tier {tier}")

    gc.collect()



--- Tier 0 ROLLING STATS ROLL3 ---
  Lysychansk: 266 feature columns
  Mariupol: 266 feature columns
  Rubizhne: 224 feature columns
  Sievierodonetsk: 266 feature columns
    impute_wide_parquet(rolling_stats): 26621521 NaN -> 16557576 NaN, was_observed_rolling_stats added
    profile -> bda_rolling_stats_roll3_t0.json + bda_rolling_stats_roll3_t0_features.json (229 features)
  Saved: bda_rolling_stats_roll3_t0.parquet (197114 rows, 269 cols, 22s)


# CELL 19: A17 -- bda_rolling_stats_roll7 (wide)

In [25]:
# @title CELL 19: bda_rolling_stats_roll7_t{tier}.parquet
def _extract_dir(label_array, ref_shape, n_buildings, tif_dir, prefix_filter=None):
    feats = {}
    if not tif_dir.exists():
        return feats, 0
    n = 0
    for tif in sorted(tif_dir.glob("*.tif")):
        stem = tif.stem
        if prefix_filter and not any(stem.startswith(p) for p in prefix_filter):
            continue
        data = read_tif_aligned(tif, ref_shape)
        stats = extract_zonal_aware(label_array, data, stem, n_buildings, agg="stat_of_stat", min_pixels=MIN_PIXELS)
        feats.update(stats)
        n += 1
    return feats, n

for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('rolling_stats_roll7', tier)
    if not FR_ROLLING_STATS and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  rolling_stats_roll7_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} ROLLING STATS ROLL7 ---")
    all_city_dfs = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue

        feats = {}
        bids = [f"{city_name}_{i}" for i in range(n_buildings)]

        bl_card_dir = STACK_ROOT / city_name / "SAR_CARD" / "temporal_stats"
        f1, n1 = _extract_dir(label_array, ref_shape, n_buildings, bl_card_dir,
                               prefix_filter=["s1__vv__baseline", "s1__vh__baseline"])
        feats.update(f1)

        bl_coh_dir = STACK_ROOT / city_name / "SAR_SLC" / "coherence_baseline"
        f2, n2 = _extract_dir(label_array, ref_shape, n_buildings, bl_coh_dir)
        feats.update(f2)

        rs_card_dir = STACK_ROOT / city_name / "temporal" / "CARD" / "rolling_stats"
        f3, n3 = _extract_dir(label_array, ref_shape, n_buildings, rs_card_dir,
                               prefix_filter=[f"s1__vv__roll7__", f"s1__vh__roll7__"])
        feats.update(f3)

        rs_coh_dir = STACK_ROOT / city_name / "temporal" / "COH" / "rolling_stats"
        f4, n4 = _extract_dir(label_array, ref_shape, n_buildings, rs_coh_dir,
                               prefix_filter=[f"s1__coh_vv__roll7__"])
        feats.update(f4)

        if not feats:
            continue

        city_df = pd.DataFrame(feats)
        city_df['building_id'] = bids
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        # NaN imputation (Plan Section 6.2)
        df = impute_wide_parquet(df, 'rolling_stats')

        out = save_v2_parquet(df, 'rolling_stats_roll7', tier)
        feat_cols = [c for c in df.columns if c not in (ID_COLS | LABEL_COLS | META_COLS_SET)]
        register_manifest('rolling_stats_roll7', 'A17', 'wide', ['city', 'building_id'], feat_cols,
                         'Q4: Rolling stats (window=7)', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No rolling stats roll7 for tier {tier}")

    gc.collect()



--- Tier 0 ROLLING STATS ROLL7 ---
  Lysychansk: 154 feature columns
  Mariupol: 154 feature columns
  Rubizhne: 112 feature columns
  Sievierodonetsk: 154 feature columns
    impute_wide_parquet(rolling_stats): 9327021 NaN -> 0 NaN, was_observed_rolling_stats added
    profile -> bda_rolling_stats_roll7_t0.json + bda_rolling_stats_roll7_t0_features.json (133 features)
  Saved: bda_rolling_stats_roll7_t0.parquet (197114 rows, 157 cols, 21s)


# CELL 20: A18 -- bda_rolling_stats_roll13 (wide)

In [26]:
# @title CELL 20: bda_rolling_stats_roll13_t{tier}.parquet
def _extract_dir(label_array, ref_shape, n_buildings, tif_dir, prefix_filter=None):
    feats = {}
    if not tif_dir.exists():
        return feats, 0
    n = 0
    for tif in sorted(tif_dir.glob("*.tif")):
        stem = tif.stem
        if prefix_filter and not any(stem.startswith(p) for p in prefix_filter):
            continue
        data = read_tif_aligned(tif, ref_shape)
        stats = extract_zonal_aware(label_array, data, stem, n_buildings, agg="stat_of_stat", min_pixels=MIN_PIXELS)
        feats.update(stats)
        n += 1
    return feats, n

for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('rolling_stats_roll13', tier)
    if not FR_ROLLING_STATS and out_path.exists():
        _sz = out_path.stat().st_size / 1e6
        print(f"  rolling_stats_roll13_t{tier}: exists ({_sz:.1f} MB), skip")
        continue

    t0 = time.time()
    print(f"\n--- Tier {tier} ROLLING STATS ROLL13 ---")
    all_city_dfs = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue

        feats = {}
        bids = [f"{city_name}_{i}" for i in range(n_buildings)]

        bl_card_dir = STACK_ROOT / city_name / "SAR_CARD" / "temporal_stats"
        f1, n1 = _extract_dir(label_array, ref_shape, n_buildings, bl_card_dir,
                               prefix_filter=["s1__vv__baseline", "s1__vh__baseline"])
        feats.update(f1)

        bl_coh_dir = STACK_ROOT / city_name / "SAR_SLC" / "coherence_baseline"
        f2, n2 = _extract_dir(label_array, ref_shape, n_buildings, bl_coh_dir)
        feats.update(f2)

        rs_card_dir = STACK_ROOT / city_name / "temporal" / "CARD" / "rolling_stats"
        f3, n3 = _extract_dir(label_array, ref_shape, n_buildings, rs_card_dir,
                               prefix_filter=[f"s1__vv__roll13__", f"s1__vh__roll13__"])
        feats.update(f3)

        rs_coh_dir = STACK_ROOT / city_name / "temporal" / "COH" / "rolling_stats"
        f4, n4 = _extract_dir(label_array, ref_shape, n_buildings, rs_coh_dir,
                               prefix_filter=[f"s1__coh_vv__roll13__"])
        feats.update(f4)

        if not feats:
            continue

        city_df = pd.DataFrame(feats)
        city_df['building_id'] = bids
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        # NaN imputation (Plan Section 6.2)
        df = impute_wide_parquet(df, 'rolling_stats')

        out = save_v2_parquet(df, 'rolling_stats_roll13', tier)
        feat_cols = [c for c in df.columns if c not in (ID_COLS | LABEL_COLS | META_COLS_SET)]
        register_manifest('rolling_stats_roll13', 'A18', 'wide', ['city', 'building_id'], feat_cols,
                         'Q4: Rolling stats (window=13)', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No rolling stats roll13 for tier {tier}")

    gc.collect()



--- Tier 0 ROLLING STATS ROLL13 ---
  Lysychansk: 154 feature columns
  Mariupol: 154 feature columns
  Rubizhne: 112 feature columns
  Sievierodonetsk: 154 feature columns
    impute_wide_parquet(rolling_stats): 9327021 NaN -> 0 NaN, was_observed_rolling_stats added
    profile -> bda_rolling_stats_roll13_t0.json + bda_rolling_stats_roll13_t0_features.json (133 features)
  Saved: bda_rolling_stats_roll13_t0.parquet (197114 rows, 157 cols, 18s)


# CELL A23 rolling_accum_coh (v29 addition for NB03e v47 R2b/P1d outputs)

A23 — `rolling_accum_coh` (long): per-pixel rolling-window matched-filter accumulators for COH VV (running_min, running_max, max_abs_delta) at N=3,7,13. Replaces methodologically-inferior A7 rolling_coh as the primary COH temporal feature; A7 retained as historical baseline for thesis ablation.


In [27]:
# @title CELL A23: bda_rolling_accum_coh_t{tier}.parquet
# Source: STACK_ROOT/{city}/temporal/COH/rolling_accum/s1__coh_vv__roll{N}__{op}__{date}.tif
# Operators: running_min, running_max, max_abs_delta
# Schema: long-format, one row per (city, building_id, date)
# Columns: s1__coh_vv__roll{N}__{op}__{stat} for stat in {mean,p10,p50,p90,std,min,max,max_abs_delta,n_pixels_valid}

ROLLING_ACCUM_OPS = ['running_min', 'running_max', 'max_abs_delta']

for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('rolling_accum_coh', tier)
    if 'FR_ROLLING_ACCUM_COH' not in dir() or (not FR_ROLLING_ACCUM_COH and out_path.exists()):
        if out_path.exists():
            _sz = out_path.stat().st_size / 1e6
            print(f"  rolling_accum_coh_t{tier}: exists ({_sz:.1f} MB), skip")
            continue

    t0 = time.time()
    print(f"\n--- Tier {tier} ROLLING ACCUM COH ---")
    all_rows = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue
        roll_dir = STACK_ROOT / city_name / "temporal" / "COH" / "rolling_accum"
        if not roll_dir.exists():
            continue

        bids = [f"{city_name}_{i}" for i in range(n_buildings)]

        # Discover dates per (window, op)
        per_date_feats = {}  # date_str -> dict of column -> array
        for tif in sorted(roll_dir.glob("s1__coh_vv__roll*__*__*.tif")):
            m = re.match(r"s1__coh_vv__roll(\d+)__([a-z_]+)__(\d{8})\.tif$", tif.name)
            if not m:
                continue
            ws, op, date_str = m.group(1), m.group(2), m.group(3)
            if op not in ROLLING_ACCUM_OPS:
                continue
            data = read_tif_aligned(tif, ref_shape)
            if 'running_min' in op:
                stats = extract_zonal_aware(label_array, data, f"s1__coh_vv__roll{ws}__{op}",
                                             n_buildings, agg="accum_min", min_pixels=MIN_PIXELS)
            elif 'running_max' in op:
                stats = extract_zonal_aware(label_array, data, f"s1__coh_vv__roll{ws}__{op}",
                                             n_buildings, agg="accum_max", min_pixels=MIN_PIXELS)
            else:  # max_abs_delta
                stats = extract_zonal_aware(label_array, data, f"s1__coh_vv__roll{ws}__{op}",
                                             n_buildings, agg="delta", min_pixels=MIN_PIXELS)
            per_date_feats.setdefault(date_str, {}).update(stats)

        if not per_date_feats:
            continue

        # build long-format rows
        for date_str, feats in per_date_feats.items():
            df_d = pd.DataFrame(feats)
            df_d['building_id'] = bids
            df_d['city'] = city_name
            df_d['date'] = date_str
            all_rows.append(df_d)

        print(f"  {city_name}: {len(per_date_feats)} dates x {n_buildings} buildings")

    if all_rows:
        df = pd.concat(all_rows, ignore_index=True)
        df = impute_wide_parquet(df, 'rolling_accum_coh')
        out = save_v2_parquet(df, 'rolling_accum_coh', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('rolling_accum_coh', 'A23', 'long',
                          ['city', 'building_id', 'date'], feat_cols,
                          'Q4b: Rolling-window matched-filter signal (COH)?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No rolling_accum_coh data for tier {tier}")
    gc.collect()


# ============================================================================
# CELL A24: bda_rolling_accum_card_t{tier}.parquet  (long format, per-date)
# ============================================================================



--- Tier 0 ROLLING ACCUM COH ---
  Lysychansk: 3 dates x 47169 buildings
  Mariupol: 5 dates x 119107 buildings
  Rubizhne: 3 dates x 16687 buildings
  Sievierodonetsk: 4 dates x 14151 buildings
    impute_wide_parquet(rolling_accum_coh): 25208106 NaN -> 0 NaN, was_observed_rolling_accum_coh added
    profile -> bda_rolling_accum_coh_t0.json + bda_rolling_accum_coh_t0_features.json (49 features)
  Saved: bda_rolling_accum_coh_t0.parquet (843707 rows, 58 cols, 13s)


# CELL A24 rolling_accum_card (v29 addition for NB03e v47 R2b/P1d outputs)

A24 — `rolling_accum_card` (long): same as A23 for CARD VV+VH backscatter.


In [28]:
# @title CELL A24: bda_rolling_accum_card_t{tier}.parquet
# Source: STACK_ROOT/{city}/temporal/CARD/rolling_accum/s1__{vv,vh}__roll{N}__{op}__{date}.tif

for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('rolling_accum_card', tier)
    if 'FR_ROLLING_ACCUM_CARD' not in dir() or (not FR_ROLLING_ACCUM_CARD and out_path.exists()):
        if out_path.exists():
            _sz = out_path.stat().st_size / 1e6
            print(f"  rolling_accum_card_t{tier}: exists ({_sz:.1f} MB), skip")
            continue

    t0 = time.time()
    print(f"\n--- Tier {tier} ROLLING ACCUM CARD ---")
    all_rows = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue
        roll_dir = STACK_ROOT / city_name / "temporal" / "CARD" / "rolling_accum"
        if not roll_dir.exists():
            continue

        bids = [f"{city_name}_{i}" for i in range(n_buildings)]
        per_date_feats = {}

        for tif in sorted(roll_dir.glob("s1__*__roll*__*__*.tif")):
            m = re.match(r"s1__(vv|vh)__roll(\d+)__([a-z_]+)__(\d{8})\.tif$", tif.name)
            if not m:
                continue
            pol, ws, op, date_str = m.group(1), m.group(2), m.group(3), m.group(4)
            if op not in ROLLING_ACCUM_OPS:
                continue
            data = read_tif_aligned(tif, ref_shape)
            prefix = f"s1__{pol}__roll{ws}__{op}"
            if 'running_min' in op:
                stats = extract_zonal_aware(label_array, data, prefix, n_buildings, agg="accum_min", min_pixels=MIN_PIXELS)
            elif 'running_max' in op:
                stats = extract_zonal_aware(label_array, data, prefix, n_buildings, agg="accum_max", min_pixels=MIN_PIXELS)
            else:
                stats = extract_zonal_aware(label_array, data, prefix, n_buildings, agg="delta", min_pixels=MIN_PIXELS)
            per_date_feats.setdefault(date_str, {}).update(stats)

        if not per_date_feats:
            continue

        for date_str, feats in per_date_feats.items():
            df_d = pd.DataFrame(feats)
            df_d['building_id'] = bids
            df_d['city'] = city_name
            df_d['date'] = date_str
            all_rows.append(df_d)
        print(f"  {city_name}: {len(per_date_feats)} dates")

    if all_rows:
        df = pd.concat(all_rows, ignore_index=True)
        df = impute_wide_parquet(df, 'rolling_accum_card')
        out = save_v2_parquet(df, 'rolling_accum_card', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('rolling_accum_card', 'A24', 'long',
                          ['city', 'building_id', 'date'], feat_cols,
                          'Q4b: Rolling-window matched-filter signal (CARD)?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No rolling_accum_card data for tier {tier}")
    gc.collect()


# ============================================================================
# CELL A25: bda_rolling_accum_ms_t{tier}.parquet  (long format, per-date)
# ============================================================================



--- Tier 0 ROLLING ACCUM CARD ---
  Lysychansk: 6 dates
  Mariupol: 12 dates
  Rubizhne: 11 dates
  Sievierodonetsk: 10 dates
    impute_wide_parquet(rolling_accum_card): 169675322 NaN -> 0 NaN, was_observed_rolling_accum_card added
    profile -> bda_rolling_accum_card_t0.json + bda_rolling_accum_card_t0_features.json (145 features)
  Saved: bda_rolling_accum_card_t0.parquet (2037365 rows, 166 cols, 118s)


# CELL A25 rolling_accum_ms (v29 addition for NB03e v47 R2b/P1d outputs)

A25 — `rolling_accum_ms` (long): same as A23/A24 for MS B11/B12/B08/B8A/NBR. Sub-question Q4b: do rolling-window accumulators outperform per-scene composites for SWIR-rise / NIR-drop damage signal?


In [29]:
# @title CELL A25: bda_rolling_accum_ms_t{tier}.parquet
# Source: STACK_ROOT/{city}/temporal/MS/rolling_accum/s2__{b11,b12,b08,b8a,nbr}__roll{N}__{op}__{date}.tif

MS_ROLLING_BANDS = ['b11', 'b12', 'b08', 'b8a', 'nbr']

for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('rolling_accum_ms', tier)
    if 'FR_ROLLING_ACCUM_MS' not in dir() or (not FR_ROLLING_ACCUM_MS and out_path.exists()):
        if out_path.exists():
            _sz = out_path.stat().st_size / 1e6
            print(f"  rolling_accum_ms_t{tier}: exists ({_sz:.1f} MB), skip")
            continue

    t0 = time.time()
    print(f"\n--- Tier {tier} ROLLING ACCUM MS ---")
    all_rows = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue
        roll_dir = STACK_ROOT / city_name / "temporal" / "MS" / "rolling_accum"
        if not roll_dir.exists():
            continue

        bids = [f"{city_name}_{i}" for i in range(n_buildings)]
        per_date_feats = {}

        for tif in sorted(roll_dir.glob("s2__*__roll*__*__*.tif")):
            m = re.match(r"s2__([a-z0-9]+)__roll(\d+)__([a-z_]+)__(\d{8})\.tif$", tif.name)
            if not m:
                continue
            band, ws, op, date_str = m.group(1), m.group(2), m.group(3), m.group(4)
            if band not in MS_ROLLING_BANDS:
                continue
            if op not in ROLLING_ACCUM_OPS:
                continue
            data = read_tif_aligned(tif, ref_shape)
            prefix = f"s2__{band}__roll{ws}__{op}"
            if 'running_min' in op:
                stats = extract_zonal_aware(label_array, data, prefix, n_buildings, agg="accum_min", min_pixels=MIN_PIXELS)
            elif 'running_max' in op:
                stats = extract_zonal_aware(label_array, data, prefix, n_buildings, agg="accum_max", min_pixels=MIN_PIXELS)
            else:
                stats = extract_zonal_aware(label_array, data, prefix, n_buildings, agg="delta", min_pixels=MIN_PIXELS)
            per_date_feats.setdefault(date_str, {}).update(stats)

        if not per_date_feats:
            continue

        for date_str, feats in per_date_feats.items():
            df_d = pd.DataFrame(feats)
            df_d['building_id'] = bids
            df_d['city'] = city_name
            df_d['date'] = date_str
            all_rows.append(df_d)
        print(f"  {city_name}: {len(per_date_feats)} dates")

    if all_rows:
        df = pd.concat(all_rows, ignore_index=True)
        df = impute_wide_parquet(df, 'rolling_accum_ms')
        out = save_v2_parquet(df, 'rolling_accum_ms', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('rolling_accum_ms', 'A25', 'long',
                          ['city', 'building_id', 'date'], feat_cols,
                          'Q4b: Rolling-window matched-filter signal (MS)?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No rolling_accum_ms data for tier {tier}")
    gc.collect()


# ============================================================================
# CELL A26: bda_block_accum_coh_t{tier}.parquet  (wide format)
# ============================================================================



--- Tier 0 ROLLING ACCUM MS ---
  Lysychansk: 12 dates
  Mariupol: 10 dates
  Rubizhne: 11 dates
  Sievierodonetsk: 13 dates
    impute_wide_parquet(rolling_accum_ms): 739648364 NaN -> 0 NaN, was_observed_rolling_accum_ms added
    profile -> bda_rolling_accum_ms_t0.json + bda_rolling_accum_ms_t0_features.json (361 features)
  Saved: bda_rolling_accum_ms_t0.parquet (2124618 rows, 409 cols, 108s)


# CELL A26 block_accum_coh (v29 addition for NB03e v47 R2b/P1d outputs)

A26 — `block_accum_coh` (wide): per-pixel block-scope matched-filter operators for COH VV per 3-month block (max_abs_delta, drop_count). P1c block_stats already covers `__min`/`__max` for COH so P1d does not duplicate them. Sub-question Q6b: block-scope matched filter vs Dietrich 7-stat block tables.


In [30]:
# @title CELL A26: bda_block_accum_coh_t{tier}.parquet
# Source: STACK_ROOT/{city}/temporal/COH/block_accum/s1__coh_vv__{block}__{op}.tif
# Operators per P1d (COH, no extrema since P1c covers __min/__max): max_abs_delta, drop_count

for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('block_accum_coh', tier)
    if 'FR_BLOCK_ACCUM_COH' not in dir() or (not FR_BLOCK_ACCUM_COH and out_path.exists()):
        if out_path.exists():
            _sz = out_path.stat().st_size / 1e6
            print(f"  block_accum_coh_t{tier}: exists ({_sz:.1f} MB), skip")
            continue

    t0 = time.time()
    print(f"\n--- Tier {tier} BLOCK ACCUM COH ---")
    all_city_dfs = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue
        blk_dir = STACK_ROOT / city_name / "temporal" / "COH" / "block_accum"
        if not blk_dir.exists():
            continue

        bids = [f"{city_name}_{i}" for i in range(n_buildings)]
        feats = {}

        for tif in sorted(blk_dir.glob("s1__coh_vv__*__*.tif")):
            m = re.match(r"s1__coh_vv__(blk\w+)__([a-z_]+)\.tif$", tif.name)
            if not m:
                continue
            block, op = m.group(1), m.group(2)
            data = read_tif_aligned(tif, ref_shape)
            prefix = f"s1__coh_vv__{block}__{op}"
            if op == 'max_abs_delta':
                stats = extract_zonal_aware(label_array, data, prefix, n_buildings, agg="delta", min_pixels=MIN_PIXELS)
            elif op in ('drop_count', 'rise_count'):
                stats = extract_zonal_aware(label_array, data, prefix, n_buildings, agg="count", min_pixels=MIN_PIXELS)
            else:
                stats = extract_zonal_aware(label_array, data, prefix, n_buildings, agg="raw", min_pixels=MIN_PIXELS)
            feats.update(stats)

        if not feats:
            continue
        city_df = pd.DataFrame(feats)
        city_df['building_id'] = bids
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        meta_cols = ['building_id', 'city']
        feat_cols_sorted = sorted([c for c in df.columns if c not in meta_cols])
        df = df[meta_cols + feat_cols_sorted]
        df = impute_wide_parquet(df, 'block_accum_coh')
        out = save_v2_parquet(df, 'block_accum_coh', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('block_accum_coh', 'A26', 'wide',
                          ['city', 'building_id'], feat_cols,
                          'Q6b: Block-scope matched-filter signal (COH)?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No block_accum_coh data for tier {tier}")
    gc.collect()


# ============================================================================
# CELL A27: bda_block_accum_card_t{tier}.parquet  (wide format)
# ============================================================================



--- Tier 0 BLOCK ACCUM COH ---
  Lysychansk: 26 feature columns
  Mariupol: 26 feature columns
  Rubizhne: 13 feature columns
  Sievierodonetsk: 13 feature columns
    impute_wide_parquet(block_accum_coh): 3188538 NaN -> 0 NaN, was_observed_block_accum_coh added
    profile -> bda_block_accum_coh_t0.json + bda_block_accum_coh_t0_features.json (23 features)
  Saved: bda_block_accum_coh_t0.parquet (197114 rows, 29 cols, 2s)


# CELL A27 block_accum_card (v29 addition for NB03e v47 R2b/P1d outputs)

A27 — `block_accum_card` (wide): same as A26 for CARD VV+VH. Includes rise_count for VH (sign-mixed signal).


In [31]:
# @title CELL A27: bda_block_accum_card_t{tier}.parquet
# Source: STACK_ROOT/{city}/temporal/CARD/block_accum/s1__{vv,vh}__{block}__{op}.tif
# Operators per P1d:
#   VV: max_abs_delta, drop_count
#   VH: max_abs_delta, drop_count, rise_count

for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('block_accum_card', tier)
    if 'FR_BLOCK_ACCUM_CARD' not in dir() or (not FR_BLOCK_ACCUM_CARD and out_path.exists()):
        if out_path.exists():
            _sz = out_path.stat().st_size / 1e6
            print(f"  block_accum_card_t{tier}: exists ({_sz:.1f} MB), skip")
            continue

    t0 = time.time()
    print(f"\n--- Tier {tier} BLOCK ACCUM CARD ---")
    all_city_dfs = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue
        blk_dir = STACK_ROOT / city_name / "temporal" / "CARD" / "block_accum"
        if not blk_dir.exists():
            continue

        bids = [f"{city_name}_{i}" for i in range(n_buildings)]
        feats = {}

        for tif in sorted(blk_dir.glob("s1__*__*__*.tif")):
            m = re.match(r"s1__(vv|vh)__(blk\w+)__([a-z_]+)\.tif$", tif.name)
            if not m:
                continue
            pol, block, op = m.group(1), m.group(2), m.group(3)
            data = read_tif_aligned(tif, ref_shape)
            prefix = f"s1__{pol}__{block}__{op}"
            if op == 'max_abs_delta':
                stats = extract_zonal_aware(label_array, data, prefix, n_buildings, agg="delta", min_pixels=MIN_PIXELS)
            elif op in ('drop_count', 'rise_count'):
                stats = extract_zonal_aware(label_array, data, prefix, n_buildings, agg="count", min_pixels=MIN_PIXELS)
            else:
                stats = extract_zonal_aware(label_array, data, prefix, n_buildings, agg="raw", min_pixels=MIN_PIXELS)
            feats.update(stats)

        if not feats:
            continue
        city_df = pd.DataFrame(feats)
        city_df['building_id'] = bids
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        meta_cols = ['building_id', 'city']
        feat_cols_sorted = sorted([c for c in df.columns if c not in meta_cols])
        df = df[meta_cols + feat_cols_sorted]
        df = impute_wide_parquet(df, 'block_accum_card')
        out = save_v2_parquet(df, 'block_accum_card', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('block_accum_card', 'A27', 'wide',
                          ['city', 'building_id'], feat_cols,
                          'Q6b: Block-scope matched-filter signal (CARD)?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No block_accum_card data for tier {tier}")
    gc.collect()


# ============================================================================
# CELL A28: bda_block_accum_ms_t{tier}.parquet  (wide format)
# ============================================================================



--- Tier 0 BLOCK ACCUM CARD ---
  Lysychansk: 60 feature columns
  Mariupol: 90 feature columns
  Rubizhne: 60 feature columns
  Sievierodonetsk: 60 feature columns
    impute_wide_parquet(block_accum_card): 9151195 NaN -> 4927850 NaN, was_observed_block_accum_card added
    profile -> bda_block_accum_card_t0.json + bda_block_accum_card_t0_features.json (76 features)
  Saved: bda_block_accum_card_t0.parquet (197114 rows, 93 cols, 7s)


# CELL A28 block_accum_ms (v29 addition for NB03e v47 R2b/P1d outputs)

A28 — `block_accum_ms` (wide): same as A26/A27 for MS B11/B12/B08/B8A/NBR. Full op set (running_min, running_max, max_abs_delta, drop_count, rise_count) since P1c does not cover MS bands.


In [32]:
# @title CELL A28: bda_block_accum_ms_t{tier}.parquet
# Source: STACK_ROOT/{city}/temporal/MS/block_accum/s2__{band}__{block}__{op}.tif
# Operators per P1d (full set since P1c does not cover MS):
#   B11/B12: running_min, running_max, max_abs_delta, rise_count
#   B08/B8A: running_min, running_max, max_abs_delta, drop_count
#   NBR:     running_min, running_max, max_abs_delta, drop_count, rise_count

MS_BLOCK_BANDS = ['b11', 'b12', 'b08', 'b8a', 'nbr']

for tier, tier_cities in TIER_CITIES.items():
    out_path = v2_path('block_accum_ms', tier)
    if 'FR_BLOCK_ACCUM_MS' not in dir() or (not FR_BLOCK_ACCUM_MS and out_path.exists()):
        if out_path.exists():
            _sz = out_path.stat().st_size / 1e6
            print(f"  block_accum_ms_t{tier}: exists ({_sz:.1f} MB), skip")
            continue

    t0 = time.time()
    print(f"\n--- Tier {tier} BLOCK ACCUM MS ---")
    all_city_dfs = []

    for city_name in tier_cities:
        label_array, ref_shape, n_buildings, _ = load_city_buildings(city_name)
        if label_array is None:
            continue
        blk_dir = STACK_ROOT / city_name / "temporal" / "MS" / "block_accum"
        if not blk_dir.exists():
            continue

        bids = [f"{city_name}_{i}" for i in range(n_buildings)]
        feats = {}

        for tif in sorted(blk_dir.glob("s2__*__*__*.tif")):
            m = re.match(r"s2__([a-z0-9]+)__(blk\w+)__([a-z_]+)\.tif$", tif.name)
            if not m:
                continue
            band, block, op = m.group(1), m.group(2), m.group(3)
            if band not in MS_BLOCK_BANDS:
                continue
            data = read_tif_aligned(tif, ref_shape)
            prefix = f"s2__{band}__{block}__{op}"
            if op == 'running_min':
                stats = extract_zonal_aware(label_array, data, prefix, n_buildings, agg="accum_min", min_pixels=MIN_PIXELS)
            elif op == 'running_max':
                stats = extract_zonal_aware(label_array, data, prefix, n_buildings, agg="accum_max", min_pixels=MIN_PIXELS)
            elif op == 'max_abs_delta':
                stats = extract_zonal_aware(label_array, data, prefix, n_buildings, agg="delta", min_pixels=MIN_PIXELS)
            elif op in ('drop_count', 'rise_count'):
                stats = extract_zonal_aware(label_array, data, prefix, n_buildings, agg="count", min_pixels=MIN_PIXELS)
            else:
                stats = extract_zonal_aware(label_array, data, prefix, n_buildings, agg="raw", min_pixels=MIN_PIXELS)
            feats.update(stats)

        if not feats:
            continue
        city_df = pd.DataFrame(feats)
        city_df['building_id'] = bids
        city_df['city'] = city_name
        all_city_dfs.append(city_df)
        print(f"  {city_name}: {len(feats)} feature columns")

    if all_city_dfs:
        df = pd.concat(all_city_dfs, ignore_index=True)
        meta_cols = ['building_id', 'city']
        feat_cols_sorted = sorted([c for c in df.columns if c not in meta_cols])
        df = df[meta_cols + feat_cols_sorted]
        df = impute_wide_parquet(df, 'block_accum_ms')
        out = save_v2_parquet(df, 'block_accum_ms', tier)
        feat_cols = [c for c in df.columns if not _is_metadata_column(c)]
        register_manifest('block_accum_ms', 'A28', 'wide',
                          ['city', 'building_id'], feat_cols,
                          'Q6b: Block-scope matched-filter signal (MS)?', tier, len(df))
        print(f"  Saved: {out.name} ({len(df)} rows, {len(df.columns)} cols, {time.time()-t0:.0f}s)")
    else:
        print(f"  No block_accum_ms data for tier {tier}")
    gc.collect()



--- Tier 0 BLOCK ACCUM MS ---
  Lysychansk: 795 feature columns
  Mariupol: 795 feature columns
  Rubizhne: 1113 feature columns
  Sievierodonetsk: 1113 feature columns
    impute_wide_parquet(block_accum_ms): 213398508 NaN -> 61696682 NaN, was_observed_block_accum_ms added
    profile -> bda_block_accum_ms_t0.json + bda_block_accum_ms_t0_features.json (1105 features)
  Saved: bda_block_accum_ms_t0.parquet (197114 rows, 1275 cols, 28s)


# FUSION PARQUETS (F1-F8)Fusions are pre-joined from atomic parquets. No ad-hoc joining at experiment time.- Long+Long fusions: outer join on `[city, building_id, date]` with `was_observed_*` flags- Long+Wide fusions: broadcast wide features to each row of long parquet- Wide+Wide fusions: inner join on `[city, building_id]`

In [33]:
# @title CELL: FUSION PARQUETS (F1-F8)
# F1: fusion_ms_card (A1+A2, long+long, date-aligned)
# F2: fusion_ms_card_cohdrop (A1+A2+A14, long+wide)
# F3: fusion_card_cohdrop (A2+A14, long+wide)
# F4: fusion_ms_cohdrop (A1+A14, long+wide)
# F5: fusion_indices_card (A5+A2, long+long)
# F6: fusion_indices_card_cohdrop (A5+A2+A14, long+long+wide)
# F7: fusion_composite_cohdrop (A9+A14, wide+wide)
# F8: fusion_composite_blockstats (A9+A15, wide+wide)

def load_v2_pq(name, tier):
    p = v2_path(name, tier)
    if not p.exists():
        return None
    return pd.read_parquet(p)

def join_long_long(df_a, df_b, name_a, name_b):
    join_cols = ['city', 'building_id', 'date']
    meta_a = [c for c in df_a.columns if c in META_COLS_SET or c in ID_COLS]
    meta_b = [c for c in df_b.columns if c in META_COLS_SET or c in ID_COLS]
    feat_a = [c for c in df_a.columns if c not in meta_a]
    feat_b = [c for c in df_b.columns if c not in meta_b]

    df = df_a.merge(df_b.drop(columns=[c for c in meta_b if c in meta_a and c not in join_cols], errors='ignore'),
                    on=join_cols, how='outer')
    df[f'was_observed_{name_a}'] = df[feat_a[0]].notna().astype(int) if feat_a else 1
    df[f'was_observed_{name_b}'] = df[feat_b[0]].notna().astype(int) if feat_b else 1
    return df

def join_long_wide(df_long, df_wide, wide_name):
    join_cols = ['city', 'building_id']
    meta_wide = [c for c in df_wide.columns if c in META_COLS_SET or c in ID_COLS]
    feat_wide = [c for c in df_wide.columns if c not in meta_wide]

    df = df_long.merge(df_wide[join_cols + feat_wide], on=join_cols, how='left')
    df[f'was_observed_{wide_name}'] = df[feat_wide[0]].notna().astype(int) if feat_wide else 0
    return df

def join_wide_wide(df_a, df_b, name_a, name_b):
    join_cols = ['city', 'building_id']
    meta_a = [c for c in df_a.columns if c in META_COLS_SET or c in ID_COLS]
    meta_b = [c for c in df_b.columns if c in META_COLS_SET or c in ID_COLS]
    feat_a = [c for c in df_a.columns if c not in meta_a]
    feat_b = [c for c in df_b.columns if c not in meta_b]

    df = df_a.merge(df_b[join_cols + feat_b], on=join_cols, how='outer')
    df[f'was_observed_{name_a}'] = df[feat_a[0]].notna().astype(int) if feat_a else 1
    df[f'was_observed_{name_b}'] = df[feat_b[0]].notna().astype(int) if feat_b else 1
    return df

FUSIONS = [
    ('fusion_ms_card',              'F1', 'long', ['scene_ms', 'scene_card'],                          'Q2: Does MS+CARD beat either alone?'),
    ('fusion_ms_card_cohdrop',      'F2', 'long', ['scene_ms', 'scene_card', 'coh_drop'],              'Q2: Full multimodal'),
    ('fusion_card_cohdrop',         'F3', 'long', ['scene_card', 'coh_drop'],                          'Q2: SAR-only multimodal'),
    ('fusion_ms_cohdrop',           'F4', 'long', ['scene_ms', 'coh_drop'],                            'Q2: Optical + COH drop'),
    ('fusion_indices_card',         'F5', 'long', ['scene_indices', 'scene_card'],                     'Q2: Indices + CARD'),
    ('fusion_indices_card_cohdrop', 'F6', 'long', ['scene_indices', 'scene_card', 'coh_drop'],         'Q2: Indices + CARD + COH drop'),
    ('fusion_composite_cohdrop',    'F7', 'wide', ['composite_prepost_bands', 'coh_drop'],             'Q2: Composite + COH drop'),
    ('fusion_composite_blockstats', 'F8', 'wide', ['composite_prepost_bands', 'block_stats'],          'Q6: Dietrich composite + block replication'),
]

for tier, tier_cities in TIER_CITIES.items():
    for fusion_name, fid, fmt, sources, question in FUSIONS:
        out_path = v2_path(fusion_name, tier)
        if not FR_FUSIONS and out_path.exists():
            _sz = out_path.stat().st_size / 1e6
            print(f"  {fusion_name}_t{tier}: exists ({_sz:.1f} MB), skip")
            continue

        t0 = time.time()
        print(f"\n--- {fusion_name} tier {tier} ---")

        dfs = {}
        missing = False
        for src in sources:
            df_src = load_v2_pq(src, tier)
            if df_src is None:
                print(f"  SKIP: {src}_t{tier} not found")
                missing = True
                break
            dfs[src] = df_src

        if missing:
            continue

        # --- Option 2: inner-join on shared city set across sources ---
        # prevents structural-NaN columns when a city is present in one source (e.g. SAR)
        # but missing in another (e.g. MS for tile-straddling cities like Borodyanka, Mykolaiv).
        # uniformly applied to all fusions; no-op for SAR-only fusions (F3) where city sets match.
        city_sets = {src: set(dfs[src]['city'].unique()) for src in sources}
        shared_cities = set.intersection(*city_sets.values())
        excluded = sorted(set.union(*city_sets.values()) - shared_cities)
        if excluded:
            for src in sources:
                dfs[src] = dfs[src][dfs[src]['city'].isin(shared_cities)].reset_index(drop=True)
            print(f"  inner-join on {len(shared_cities)} shared cities (excluded: {excluded})")

        # determine join types
        is_long = {}
        for src in sources:
            is_long[src] = 'date' in dfs[src].columns

        # build fusion
        result = None
        for i, src in enumerate(sources):
            if i == 0:
                result = dfs[src]
                continue
            if is_long.get(sources[0]) and is_long.get(src):
                result = join_long_long(result, dfs[src], sources[0], src)
            elif is_long.get(sources[0]) and not is_long.get(src):
                result = join_long_wide(result, dfs[src], src)
            elif not is_long.get(sources[0]) and not is_long.get(src):
                result = join_wide_wide(result, dfs[src], sources[0], src)
            else:
                result = join_long_wide(dfs[src], result, sources[0])

        if result is not None and len(result) > 0:
            out = save_v2_parquet(result, fusion_name, tier)
            feat_cols = [c for c in result.columns if not _is_metadata_column(c)]
            register_manifest(fusion_name, fid, fmt, ['city', 'building_id'] + (['date'] if 'date' in result.columns else []),
                             feat_cols, question, tier, len(result), composed_of=sources,
                             cities_excluded=excluded if excluded else None)
            print(f"  Saved: {out.name} ({len(result)} rows, {len(result.columns)} cols, {time.time()-t0:.0f}s)")
        else:
            print(f"  Empty fusion for {fusion_name} tier {tier}")

        gc.collect()



--- fusion_ms_card tier 0 ---
    profile -> bda_fusion_ms_card_t0.json + bda_fusion_ms_card_t0_features.json (115 features)
  Saved: bda_fusion_ms_card_t0.parquet (4622080 rows, 134 cols, 45s)

--- fusion_ms_card_cohdrop tier 0 ---
  inner-join on 3 shared cities (excluded: ['Rubizhne'])
    profile -> bda_fusion_ms_card_cohdrop_t0.json + bda_fusion_ms_card_cohdrop_t0_features.json (147 features)
  Saved: bda_fusion_ms_card_cohdrop_t0.parquet (4238279 rows, 179 cols, 53s)

--- fusion_card_cohdrop tier 0 ---
  inner-join on 3 shared cities (excluded: ['Rubizhne'])
    profile -> bda_fusion_card_cohdrop_t0.json + bda_fusion_card_cohdrop_t0_features.json (48 features)
  Saved: bda_fusion_card_cohdrop_t0.parquet (2214662 rows, 68 cols, 12s)

--- fusion_ms_cohdrop tier 0 ---
  inner-join on 3 shared cities (excluded: ['Rubizhne'])
    profile -> bda_fusion_ms_cohdrop_t0.json + bda_fusion_ms_cohdrop_t0_features.json (129 features)
  Saved: bda_fusion_ms_cohdrop_t0.parquet (2117955 rows, 15

# CELL: GROUPKFOLD

In [34]:
# @title CELL: GROUPKFOLD ASSIGNMENT
import stack_groupkfold
importlib.reload(stack_groupkfold)
from stack_groupkfold import run as run_groupkfold

v2_gkf_path = V2_DIR / "groupkfold.parquet"

if FR_GROUPKFOLD or not v2_gkf_path.exists():
    GKF = run_groupkfold(stack_root=STACK_ROOT, n_folds=None, output_path=v2_gkf_path)
else:
    print(f"  GroupKFold exists, skipping")


GROUPKFOLD: CROSS-VALIDATION FOLD ASSIGNMENT BY CITY

  Total cities: 21
  ML-ready cities: 21
  Fold mode: leave_one_city_out
  N folds: 21

  City                   Fold    Bldg    Dmg  CARD  COH   MS
  ---------------------- ----  ------  -----  ----  ----  ----
  Kharkiv                  0   213845    238    Y     -     Y
  Mykolaiv                 1   137326     84    Y     Y     -
  Mariupol                 2   119107   3055    Y     Y     Y
  Chernihiv                3    58626    333    Y     Y     Y
  Kramatorsk               4    58518     19    Y     Y     Y
  Borodyanka               5    49969     62    Y     Y     -
  Lysychansk               6    47169   1071    Y     Y     Y
  Okhtyrka                 7    27603     35    Y     Y     Y
  Moschun                  8    20565     73    Y     Y     Y
  Hostomel                 9    19798    511    Y     Y     Y
  Trostianets             10    18656     18    Y     -     Y
  Irpin                   11    17686    249    Y   

# CELL: WRITE MANIFEST

In [35]:
# @title CELL: WRITE parquet_manifest.json
from datetime import datetime as _dt

# bugfix: register on-disk parquets that were skipped this run (incl. prior-tier runs)
scan_disk_and_register()

manifest_out = {
    'version': 'v2',
    'created': _dt.now().isoformat(),
    'created_by': 'NB05b v22',
    'tier_selection': TIER_SELECTION,
    'min_pixels': MIN_PIXELS,
    'nan_handling': 'global_median_impute + was_observed flags for residual NaN',
    'parquets': MANIFEST_ENTRIES,
}

manifest_path = V2_DIR / 'parquet_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest_out, f, indent=2, default=str)

print(f"Manifest written: {manifest_path}")
print(f"  {len(MANIFEST_ENTRIES)} parquet definitions")
for name, info in sorted(MANIFEST_ENTRIES.items()):
    tiers = info.get('tiers_built', [])
    n_feat = info.get('n_features', 0)
    print(f"  {name:<35s} [{info['id']}] {info['format']:>5s} {n_feat:>4d} features, tiers={tiers}")


  disk-scan: registered bda_block_stats_t1.parquet (265550 rows, 180 features)
  disk-scan: registered bda_block_stats_t2.parquet (444707 rows, 894 features)
  disk-scan: registered bda_buildings_t0.parquet (197114 rows, 0 features)
  disk-scan: registered bda_buildings_t1.parquet (265550 rows, 0 features)
  disk-scan: registered bda_buildings_t2.parquet (444707 rows, 0 features)
  disk-scan: registered bda_card_drop_t1.parquet (265550 rows, 14 features)
  disk-scan: registered bda_card_drop_t2.parquet (444707 rows, 14 features)
  disk-scan: registered bda_coh_drop_t1.parquet (265550 rows, 13 features)
  disk-scan: registered bda_coh_drop_t2.parquet (444707 rows, 13 features)
  disk-scan: registered bda_composite_prepost_bands_t1.parquet (215581 rows, 190 features)
  disk-scan: registered bda_composite_prepost_bands_t2.parquet (307381 rows, 190 features)
  disk-scan: registered bda_composite_prepost_landuse_t1.parquet (215581 rows, 5 features)
  disk-scan: registered bda_composite_prep

# CELL: VALIDATE ALL v2 PARQUETS

In [36]:
# @title CELL: VALIDATE v2 PARQUETS
print("=" * 70)
print("VALIDATE: CHECK ALL v2 PER-TIER PARQUETS")
print("=" * 70)

parquet_files = sorted(V2_DIR.glob("bda_*_t*.parquet"))
print(f"\n  Per-tier parquets found: {len(parquet_files)}")

for pq in parquet_files:
    df = pd.read_parquet(pq)
    n_cities = df['city'].nunique() if 'city' in df.columns else 0
    n_rows = len(df)
    n_cols = len(df.columns)
    size_mb = pq.stat().st_size / 1e6

    warn = ""
    if 'date' not in df.columns and 'date1' not in df.columns:
        feat_cols = [c for c in df.columns if c not in ('building_id', 'city')]
        struct_nan = 0
        for col in feat_cols:
            nan_cities = set()
            ok_cities = set()
            for city, grp in df.groupby('city'):
                if grp[col].isna().all():
                    nan_cities.add(city)
                elif grp[col].notna().any():
                    ok_cities.add(city)
            if nan_cities and ok_cities:
                struct_nan += 1
        if struct_nan > 0:
            warn = f" WARNING: {struct_nan} structural NaN cols!"

    print(f"  {pq.name:<55s} {n_rows:>8d} rows  {n_cols:>4d} cols  {n_cities:>2d} cities  {size_mb:>6.1f} MB{warn}")

# validate manifest completeness
manifest_path = V2_DIR / 'parquet_manifest.json'
if manifest_path.exists():
    with open(manifest_path) as f:
        mf = json.load(f)
    print(f"\n  Manifest: {len(mf['parquets'])} entries")
    for name, info in mf['parquets'].items():
        for tier in TIER_SELECTION:
            pq_path = V2_DIR / f"bda_{name}_t{tier}.parquet"
            status = "OK" if pq_path.exists() else "MISSING"
            if status == "MISSING":
                print(f"  WARNING: {pq_path.name} in manifest but {status}")
else:
    print("  WARNING: parquet_manifest.json not found!")

print(f"\n{'='*70}")
print("VALIDATE COMPLETE")
print(f"{'='*70}")


VALIDATE: CHECK ALL v2 PER-TIER PARQUETS

  Per-tier parquets found: 99
  bda_block_accum_card_t0.parquet                           197114 rows    93 cols   4 cities    32.7 MB WARNING: 5 structural NaN cols!
  bda_block_accum_coh_t0.parquet                            197114 rows    29 cols   4 cities     7.7 MB WARNING: 4 structural NaN cols!
  bda_block_accum_ms_t0.parquet                             197114 rows  1275 cols   4 cities    36.5 MB WARNING: 84 structural NaN cols!
  bda_block_stats_t0.parquet                                197114 rows   745 cols   4 cities   348.1 MB WARNING: 38 structural NaN cols!
  bda_block_stats_t1.parquet                                265550 rows   183 cols  11 cities   205.6 MB
  bda_block_stats_t2.parquet                                444707 rows   897 cols   6 cities   577.2 MB
  bda_buildings_t0.parquet                                  197114 rows    26 cols   4 cities     7.2 MB WARNING: 1 structural NaN cols!
  bda_buildings_t1.parquet     

In [37]:
# @title DIAGNOSTIC: plan (planned/downloaded/extracted) + disk + composite input counts
TARGETS = ['Borodyanka', 'Bucha', 'Mykolaiv', 'Kharkiv']

plan_path = STACK_ROOT / "nb02a_scene_plan.json"
with open(plan_path) as f:
    plan = json.load(f)
print(f"Plan top-level keys: {len(plan)} entries (cities at top level)")
sample = next(iter(plan))
print(f"Plan city-entry keys (sample '{sample}'): {list(plan[sample].keys())}")

for city in TARGETS:
    print(f"\n{'='*70}")
    print(f"CITY: {city}")
    print('='*70)
    cp = plan.get(city, {})
    if not cp:
        print("  NOT in plan")
        continue

    # ---- PLAN: MS-related sections ----
    ms_keys = [k for k in cp.keys() if any(t in k.lower() for t in ('ms', 's2', 'optical', 'multispec', 'sentinel2', 'sentinel-2'))]
    print(f"  plan MS-keys: {ms_keys}")
    for k in ms_keys:
        dump = json.dumps(cp[k], indent=4, default=str)
        print(f"\n  plan[{k}] =")
        print(dump if len(dump) < 3500 else dump[:3500] + "\n    ... (truncated)")

    # ---- DISK: multispectral/ tree ----
    ms_root = STACK_ROOT / city / "multispectral"
    print(f"\n  DISK {ms_root}:")
    if not ms_root.exists():
        print("    MISSING")
    else:
        for sub in sorted(ms_root.iterdir()):
            if sub.is_dir():
                children = list(sub.iterdir())
                n_tifs = sum(1 for p in children if p.suffix == '.tif')
                subdirs = sorted([p for p in children if p.is_dir()])
                print(f"    {sub.name}/   {len(subdirs)} subdirs, {n_tifs} loose tifs")
                for sd in subdirs:
                    sd_tifs = list(sd.glob("*.tif"))
                    print(f"      {sd.name}/   {len(sd_tifs)} tifs")
                    for t in sd_tifs[:3]:
                        print(f"        {t.name}")
                    if len(sd_tifs) > 3:
                        print(f"        ... (+{len(sd_tifs)-3} more)")
            else:
                print(f"    {sub.name}")

    # ---- COMPOSITE INPUT SCENE COUNTS (per period, pre/post battle) ----
    from collections import Counter
    import re as _re
    scene_tifs = []
    for sub in ('scenes', 'clipped', 'processed', 'raw', 'L2A', 'l2a'):
        d = ms_root / sub
        if d.exists():
            scene_tifs = list(d.glob("**/*.tif"))
            print(f"\n  Scene source: {sub}/   ({len(scene_tifs)} tifs)")
            break
    if not scene_tifs:
        scene_tifs = [p for p in ms_root.rglob("*.tif") if 'composites' not in p.parts]
        print(f"\n  Scene source: rglob fallback   ({len(scene_tifs)} tifs outside composites/)")

    dates = []
    for t in scene_tifs:
        m = _re.search(r'(\d{8})', t.stem)
        if m:
            dates.append(m.group(1))
    uniq_dates = sorted(set(dates))
    print(f"    unique acquisition dates: {len(uniq_dates)}")
    if uniq_dates:
        print(f"    range: {uniq_dates[0]} -- {uniq_dates[-1]}")
        bs = city_meta.get(city, {}).get('battle_start', '')
        be = city_meta.get(city, {}).get('battle_stop', '')
        if bs:
            bs8 = bs.replace('-', '')[:8]
            be8 = be.replace('-', '')[:8] if be else None
            pre = [d for d in uniq_dates if d < bs8]
            post = [d for d in uniq_dates if (be8 and d > be8) or (not be8 and d >= bs8)]
            cross = [d for d in uniq_dates if d not in pre and d not in post]
            print(f"    battle: {bs[:10]} -- {be[:10] if be else 'ongoing'}")
            print(f"    pre-battle dates:   {len(pre)}")
            print(f"    post-battle dates:  {len(post)}")
            print(f"    cross-battle dates: {len(cross)}")

Plan top-level keys: 50 entries (cities at top level)
Plan city-entry keys (sample 'Avdiivka'): ['city', 'tier', 'orbit', 'battle_start', 'battle_stop', 'slc', 'card', 'ms', 'counts']

CITY: Borodyanka
  plan MS-keys: ['ms']

  plan[ms] =
[]

  DISK /mnt/f/PROJECTS/masterthesis/data_stack/Borodyanka/multispectral:
    MISSING

  Scene source: rglob fallback   (0 tifs outside composites/)
    unique acquisition dates: 0

CITY: Bucha
  plan MS-keys: ['ms']

  plan[ms] =
[
    {
        "date": "2020-12-06",
        "period": "pre_battle",
        "status": "on_disk",
        "modality": "ms",
        "scene_id": "b3d92fe6-1c38-4b26-b46c-29fad5064bff",
        "scene_name": "S2A_MSIL2A_20201206T091351_N0500_R050_T35UQS_20230304T190542.SAFE",
        "cloud_cover": 7.577983000000001,
        "tile_id": "T35UQS",
        "coverage_pct": 99.99999999999993
    },
    {
        "date": "2021-01-20",
        "period": "pre_battle",
        "status": "on_disk",
        "modality": "ms",
        

In [38]:
# @title DIAGNOSTIC: why did NB02a produce empty MS list for Borodyanka + Mykolaiv
import os
from datetime import datetime as _dt

TARGETS = ['Borodyanka', 'Bucha', 'Mykolaiv', 'Kharkiv']

plan_path = STACK_ROOT / "nb02a_scene_plan.json"
print(f"plan_path: {plan_path}")
print(f"plan mtime: {_dt.fromtimestamp(plan_path.stat().st_mtime).isoformat()}")
print(f"plan size:  {plan_path.stat().st_size / 1e6:.2f} MB\n")

with open(plan_path) as f:
    plan = json.load(f)

for city in TARGETS:
    cp = plan.get(city, {})
    print(f"\n=== {city} ===")
    print(f"  top-level keys: {list(cp.keys())}")
    print(f"  tier:         {cp.get('tier')}")
    print(f"  orbit:        {cp.get('orbit')}")
    print(f"  battle:       {cp.get('battle_start')} -- {cp.get('battle_stop')}")
    print(f"  counts:       {cp.get('counts')}")
    print(f"  n_slc:        {len(cp.get('slc', []))}")
    print(f"  n_card:       {len(cp.get('card', []))}")
    print(f"  n_ms:         {len(cp.get('ms', []))}")

    # any non-ms keys that could hold rejected/candidate/excluded info
    misc = {k: v for k, v in cp.items() if k not in ('city','tier','orbit','battle_start','battle_stop','slc','card','ms','counts')}
    if misc:
        print(f"  OTHER keys (candidates/excluded/log?):")
        for k, v in misc.items():
            if isinstance(v, list):
                print(f"    {k}: list len={len(v)}")
                if v:
                    print(f"      [0] = {json.dumps(v[0], indent=6, default=str)[:400]}")
            elif isinstance(v, dict):
                print(f"    {k}: dict keys={list(v.keys())}")
                print(f"      = {json.dumps(v, indent=6, default=str)[:400]}")
            else:
                print(f"    {k}: {v}")

# ---- look for any NB02a discovery/rejection log ----
print(f"\n{'='*70}\nSearch for NB02a auxiliary logs:\n{'='*70}")
candidates = [
    STACK_ROOT / "nb02a_scene_discovery.json",
    STACK_ROOT / "nb02a_scene_rejected.json",
    STACK_ROOT / "nb02a_ms_candidates.json",
    STACK_ROOT / "nb02a_log.json",
    STACK_ROOT / "nb02a_plan_build.log",
]
for p in candidates:
    if p.exists():
        print(f"  FOUND: {p}  ({p.stat().st_size/1e3:.1f} KB)")
    else:
        print(f"  absent: {p.name}")

# also scan STACK_ROOT for any nb02a_* file
aux = sorted(STACK_ROOT.glob("nb02a_*"))
print(f"\n  All nb02a_* in STACK_ROOT: {[p.name for p in aux]}")

# check per-city for any ms discovery artifact
print(f"\n{'='*70}\nPer-city MS discovery artifacts:\n{'='*70}")
for city in TARGETS:
    city_root = STACK_ROOT / city
    hits = []
    for pat in ('*ms*.json', '*s2*.json', '*candidate*.json', '*discovery*.json', '*rejected*.json'):
        hits.extend(city_root.glob(pat))
    print(f"  {city}: {[p.name for p in hits] if hits else 'none'}")

plan_path: /mnt/f/PROJECTS/masterthesis/data_stack/nb02a_scene_plan.json
plan mtime: 2026-04-23T01:26:28
plan size:  2.13 MB


=== Borodyanka ===
  top-level keys: ['city', 'tier', 'orbit', 'battle_start', 'battle_stop', 'slc', 'card', 'ms', 'counts', 'coh_pairs']
  tier:         2
  orbit:        87
  battle:       2022-02-28 -- 2022-04-01
  counts:       {'slc_total': 10, 'slc_on_disk': 10, 'slc_to_download': 0, 'card_total': 10, 'card_on_disk': 10, 'card_to_download': 0, 'card_disk_actual': 19, 'ms_total': 0, 'ms_on_disk': 0, 'ms_to_download': 0}
  n_slc:        10
  n_card:       10
  n_ms:         0
  OTHER keys (candidates/excluded/log?):
    coh_pairs: list len=9
      [0] = {
      "date_a": "2021-12-21",
      "date_b": "2022-01-02",
      "timestamp": "2026-04-09T23:31:54.398408"
}

=== Bucha ===
  top-level keys: ['city', 'tier', 'orbit', 'battle_start', 'battle_stop', 'slc', 'card', 'ms', 'counts', 'coh_pairs']
  tier:         2
  orbit:        160
  battle:       2022-02-2

In [39]:
# @title DIAGNOSTIC: compare AOI + cities_config (no per-feature iteration)
import geopandas as gpd

TARGETS = ['Borodyanka', 'Bucha', 'Mykolaiv', 'Kharkiv']

# ---- per-city AOI.geojson (summary only + single aoi row via load_aoi) ----
print("="*70)
print("AOI.geojson summary per city")
print("="*70)
aoi_rows = {}
for city in TARGETS:
    aoi_path = STACK_ROOT / city / "AOI.geojson"
    if not aoi_path.exists():
        print(f"\n{city}: AOI.geojson MISSING at {aoi_path}")
        continue
    gdf = gpd.read_file(aoi_path)
    bounds = gdf.total_bounds
    print(f"\n{city}  ({aoi_path.name})")
    print(f"  n_features: {len(gdf)}   CRS: {gdf.crs}")
    print(f"  bounds:     {bounds}")
    # columns present in geojson
    cols = [c for c in gdf.columns if c != 'geometry']
    print(f"  columns:    {cols}")
    # use project helper to get the single AOI meta row (aoi_bbox/city row)
    try:
        aoi_row = load_aoi(city)
        attrs = {k: v for k, v in dict(aoi_row).items() if k != 'geometry'}
        aoi_rows[city] = attrs
        print(f"  load_aoi() row (single, no buildings):")
        for k, v in attrs.items():
            print(f"    {k!r:20s} = {v!r}")
    except Exception as e:
        print(f"  load_aoi() failed: {e}")

# ---- attribute key + value diff on the load_aoi row ----
print(f"\n{'='*70}\nAOI meta-row key diff (missing vs working)\n{'='*70}")
miss_keys = set(aoi_rows.get('Borodyanka', {}).keys()) | set(aoi_rows.get('Mykolaiv', {}).keys())
work_keys = set(aoi_rows.get('Bucha', {}).keys()) | set(aoi_rows.get('Kharkiv', {}).keys())
print(f"  only in MISSING: {sorted(miss_keys - work_keys)}")
print(f"  only in WORKING: {sorted(work_keys - miss_keys)}")
for k in sorted(miss_keys & work_keys):
    vals = {city: aoi_rows.get(city, {}).get(k) for city in TARGETS}
    miss_vals = {vals.get('Borodyanka'), vals.get('Mykolaiv')}
    work_vals = {vals.get('Bucha'), vals.get('Kharkiv')}
    if miss_vals != work_vals:
        print(f"\n  key={k!r} DIFFERS:")
        for city in TARGETS:
            print(f"    {city:15s} = {vals.get(city)!r}")

# ---- cities_config.json ----
print(f"\n{'='*70}\ncities_config.json\n{'='*70}")
cfg_candidates = [
    NOTEBOOKS_DIR / "cities_config.json",
    NOTEBOOKS_DIR.parent / "data" / "cities_config.json",
    STACK_ROOT / "cities_config.json",
]
cfg = None
cfg_path = None
for p in cfg_candidates:
    if p.exists():
        cfg_path = p
        with open(p) as f:
            cfg = json.load(f)
        break
print(f"  path: {cfg_path}")
if cfg is None:
    print("  NOT FOUND in any expected location")
else:
    if isinstance(cfg, list):
        cfg_by_city = {c.get('city', c.get('name')): c for c in cfg if isinstance(c, dict)}
    elif isinstance(cfg, dict):
        if 'cities' in cfg and isinstance(cfg['cities'], (list, dict)):
            inner = cfg['cities']
            cfg_by_city = {c.get('city', c.get('name')): c for c in inner} if isinstance(inner, list) else inner
        else:
            cfg_by_city = cfg
    print(f"  total cities: {len(cfg_by_city)}")
    print(f"  sample keys:  {list(next(iter(cfg_by_city.values())).keys()) if cfg_by_city else []}")
    for city in TARGETS:
        entry = cfg_by_city.get(city)
        print(f"\n  {city}:")
        if entry is None:
            print(f"    NOT in config")
        else:
            for k, v in entry.items():
                print(f"    {k!r:25s} = {v!r}")

AOI.geojson summary per city

Borodyanka  (AOI.geojson)
  n_features: 49971   CRS: EPSG:4326
  bounds:     [29.62520595 50.54152844 30.07474683 50.80507973]
  columns:    ['id', 'names', 'sources', 'height', 'level', 'is_underground', 'num_floors', 'roof_shape', 'subtype', 'class', 'has_parts', 'version', 'damage', 'damage_label', 'damage_binary', 'ems98_grade', 'unosat_id', 'unosat_date', 'unosat_ep', 'match_method', 'match_distance', 'feature_type', 'city_name', 'in_dietrich_aoi', 'utm_epsg', 'utm_minx', 'utm_miny', 'utm_maxx', 'utm_maxy', 'width_px', 'height_px', 'divisor', 'resolution_m', 'oblast', 'tier', 'priority', 'admin_level', 'battle_start', 'battle_stop']
  load_aoi() row (single, no buildings):
    'feature_type'       = 'city_polygon'
    'city_name'          = 'Borodyanka'
    'utm_epsg'           = None
    'utm_minx'           = None
    'utm_miny'           = None
    'utm_maxx'           = None
    'utm_maxy'           = None
    'width_px'           = None
    'heig

In [40]:
# @title DIAGNOSTIC: AOI bbox size + S2 tile coverage for MS discovery
import geopandas as gpd
from shapely.geometry import box

TARGETS = ['Borodyanka', 'Bucha', 'Mykolaiv', 'Kharkiv']

def summarize_feature(gdf, feature_type):
    sub = gdf[gdf.get('feature_type', '') == feature_type]
    if len(sub) == 0:
        return None
    row = sub.iloc[0]
    geom4326 = sub.to_crs(4326).geometry.iloc[0]
    minx, miny, maxx, maxy = geom4326.bounds
    # estimate km-scale bbox
    mean_lat = (miny + maxy) / 2
    km_lon = (maxx - minx) * 111.32 * np.cos(np.radians(mean_lat))
    km_lat = (maxy - miny) * 110.57
    # area in projected units if possible
    utm_epsg_val = row.get('utm_epsg')
    area_km2 = None
    if utm_epsg_val and str(utm_epsg_val).lower() not in ('none', 'nan'):
        try:
            geom_utm = sub.to_crs(int(float(utm_epsg_val))).geometry.iloc[0]
            area_km2 = geom_utm.area / 1e6
        except Exception:
            pass
    return {
        'bounds_4326': (round(minx,4), round(miny,4), round(maxx,4), round(maxy,4)),
        'bbox_km':     (round(km_lon,2), round(km_lat,2)),
        'area_km2':    round(area_km2,2) if area_km2 else None,
        'utm_epsg':    row.get('utm_epsg'),
        'admin_level': row.get('admin_level'),
        'tier':        row.get('tier'),
        'geom_4326':   geom4326,
    }

# ---- scan per-city ----
results = {}
for city in TARGETS:
    aoi_path = STACK_ROOT / city / "AOI.geojson"
    if not aoi_path.exists():
        print(f"{city}: AOI.geojson MISSING"); continue
    gdf = gpd.read_file(aoi_path)
    types = sorted(gdf['feature_type'].dropna().unique().tolist()) if 'feature_type' in gdf.columns else []
    print(f"\n{city}: feature_types = {types}")
    results[city] = {}
    for ft in ('aoi_bbox', 'city_polygon'):
        info = summarize_feature(gdf, ft)
        results[city][ft] = info
        if info:
            print(f"  {ft}:")
            for k, v in info.items():
                if k != 'geom_4326':
                    print(f"    {k:15s} = {v}")

# ---- ratio + verdict ----
print(f"\n{'='*70}")
print("HYPOTHESIS CHECK: S2 tile is 100 km x 100 km. Single-tile coverage is")
print("unlikely above ~85 km bbox; NB02a likely needs ~99% single-tile coverage.")
print('='*70)
for city in TARGETS:
    r = results.get(city, {})
    bb = r.get('aoi_bbox') or r.get('city_polygon')
    if not bb:
        continue
    kmx, kmy = bb['bbox_km']
    max_km = max(kmx, kmy)
    ms_planned = {'Borodyanka':0,'Bucha':14,'Mykolaiv':0,'Kharkiv':15}[city]
    verdict = "OK (< 85 km)" if max_km < 85 else "SUSPECT (>= 85 km, likely straddles S2 tiles)"
    src = 'aoi_bbox' if r.get('aoi_bbox') else 'city_polygon (aoi_bbox missing!)'
    print(f"  {city:12s}  source={src:30s}  max_extent={max_km:6.2f} km  ms_planned={ms_planned}  -> {verdict}")

# ---- count distinct S2 tiles actually present in plan for working cities (reference) ----
print(f"\n{'='*70}\nS2 tile_ids seen in plan (from scenes already on disk):\n{'='*70}")
with open(STACK_ROOT / "nb02a_scene_plan.json") as f:
    plan = json.load(f)
for city in TARGETS:
    cp = plan.get(city, {})
    tiles = sorted({s.get('tile_id') for s in cp.get('ms', []) if s.get('tile_id')})
    print(f"  {city:12s}: {tiles if tiles else '(none planned)'}")


Borodyanka: feature_types = ['aoi_bbox', 'building', 'city_polygon']
  aoi_bbox:
    bounds_4326     = (29.6252, 50.5415, 30.0747, 50.8051)
    bbox_km         = (np.float64(31.71), 29.14)
    area_km2        = 865.08
    utm_epsg        = 32635.0
    admin_level     = nan
    tier            = nan
  city_polygon:
    bounds_4326     = (29.641, 50.5538, 30.0608, 50.7946)
    bbox_km         = (np.float64(29.61), 26.62)
    area_km2        = None
    utm_epsg        = nan
    admin_level     = ADM3
    tier            = 1.0

Bucha: feature_types = ['aoi_bbox', 'building', 'city_polygon']
  aoi_bbox:
    bounds_4326     = (30.154, 50.5096, 30.2663, 50.5811)
    bbox_km         = (np.float64(7.95), 7.91)
    area_km2        = 58.98
    utm_epsg        = 32636.0
    admin_level     = nan
    tier            = nan
  city_polygon:
    bounds_4326     = (30.1593, 50.5165, 30.2595, 50.5748)
    bbox_km         = (np.float64(7.09), 6.45)
    area_km2        = None
    utm_epsg        = nan
   

In [41]:
# @title DIAGNOSTIC: inspect per-city MS metadata + ms_scene_discovery module
TARGETS = ['Borodyanka', 'Bucha', 'Mykolaiv', 'Kharkiv']

# where does NB02a put per-city MS metadata? use the global that NB02a reads from
print(f"MS_METADATA_DIR: {MS_METADATA_DIR}")
print(f"  exists: {MS_METADATA_DIR.exists()}")
if MS_METADATA_DIR.exists():
    top = sorted(MS_METADATA_DIR.iterdir())
    print(f"  top entries ({len(top)}):")
    for e in top[:30]:
        print(f"    {e.name}{'/' if e.is_dir() else ''}")

# per-city: find any JSON named with the city
for city in TARGETS:
    print(f"\n--- {city} ---")
    # direct city dir
    city_dir = MS_METADATA_DIR / city
    if city_dir.exists():
        files = sorted(city_dir.iterdir())
        print(f"  {city_dir}: {len(files)} files")
        for f in files:
            print(f"    {f.name}  ({f.stat().st_size/1e3:.1f} KB)")
    # flat files prefixed with city name
    flat = sorted(MS_METADATA_DIR.glob(f"{city}*"))
    flat += sorted(MS_METADATA_DIR.glob(f"*{city}*"))
    flat = sorted(set(flat))
    if flat:
        print(f"  flat matches in {MS_METADATA_DIR.name}/:")
        for f in flat:
            print(f"    {f.name}  ({f.stat().st_size/1e3:.1f} KB)")

    # also look inside STACK_ROOT/city for any MS metadata
    city_root = STACK_ROOT / city
    for pat in ('*ms*.json', '*s2*.json', 'ms_*.json'):
        hits = list(city_root.glob(pat)) + list(city_root.rglob(pat))
        for h in set(hits):
            if 'composites' not in h.parts and 'flat' not in h.parts:
                print(f"    stack_root hit: {h.relative_to(STACK_ROOT)}  ({h.stat().st_size/1e3:.1f} KB)")

# dump the ms_scene_discovery module source (filter logic lives here)
print(f"\n{'='*70}\nms_scene_discovery module source\n{'='*70}")
import ms_scene_discovery, inspect
src_path = Path(inspect.getsourcefile(ms_scene_discovery))
print(f"  path: {src_path}")
print(f"  size: {src_path.stat().st_size/1e3:.1f} KB")
src_text = src_path.read_text()
print(f"  lines: {len(src_text.splitlines())}")
# show only the relevant functions -- search for coverage_pct, cloud_cover, filter, reject
import re as _re
for fn_match in _re.finditer(r'^def (\w+)\(', src_text, _re.M):
    name = fn_match.group(1)
    start = fn_match.start()
    # find end of function (next def at same indent or EOF)
    tail = src_text[start:]
    next_def = _re.search(r'\n(def |class )', tail[1:])
    body = tail[:next_def.start()+1] if next_def else tail
    if any(tok in body for tok in ('coverage_pct', 'cloud_cover', 'MIN_COVERAGE', 'CLOUD_MAX', 'reject', 'SAR_WINDOW', 'min_scenes')):
        print(f"\n--- def {name} ---")
        print(body[:3000])
        if len(body) > 3000:
            print(f"    ... ({len(body)} chars total)")

MS_METADATA_DIR: /content/drive_f/masterthesis/data/satellite/MS/metadata
  exists: True
  top entries (52):
    Avdiivka_ms_scene_metadata.json
    Bakhmut_ms_scene_metadata.json
    Balakliia_ms_scene_metadata.json
    Borodyanka_ms_scene_metadata.json
    Bucha_ms_scene_metadata.json
    Chasiv Yar_ms_scene_metadata.json
    Chernihiv_ms_scene_metadata.json
    Chornobaivka_ms_scene_metadata.json
    Chuhuiv_ms_scene_metadata.json
    Dmytrivka_ms_scene_metadata.json
    Dnipro_ms_scene_metadata.json
    Hirske_ms_scene_metadata.json
    Hostomel_ms_scene_metadata.json
    Huliaipole_ms_scene_metadata.json
    Irpin_ms_scene_metadata.json
    Izyum_ms_scene_metadata.json
    Kharkiv_ms_scene_metadata.json
    Kherson_ms_scene_metadata.json
    Kramatorsk_ms_scene_metadata.json
    Krasnohorivka_ms_scene_metadata.json
    Kryvyi Rih_ms_scene_metadata.json
    Kupiansk_ms_scene_metadata.json
    Kurakhove_ms_scene_metadata.json
    Lviv_ms_scene_metadata.json
    Lysychansk_ms_scene_m

In [42]:
# @title DIAGNOSTIC: read metadata + reproduce coverage calculation
TARGETS = ['Borodyanka', 'Bucha', 'Mykolaiv', 'Kharkiv']

# ---- (1) dump metadata files ----
print("="*70)
print("PER-CITY MS METADATA FILE CONTENTS")
print("="*70)
for city in TARGETS:
    mp = MS_METADATA_DIR / f"{city}_ms_scene_metadata.json"
    if not mp.exists():
        print(f"\n{city}: metadata missing"); continue
    with open(mp) as f:
        meta = json.load(f)
    print(f"\n--- {city} ({mp.stat().st_size/1e3:.1f} KB) ---")
    print(f"  top keys: {list(meta.keys()) if isinstance(meta, dict) else 'list'}")
    dump = json.dumps(meta, indent=2, default=str)
    print(dump if len(dump) < 6000 else dump[:6000] + "\n  ... (truncated)")

# ---- (2) reproduce the CDSE query + coverage calc for the two missing cities ----
print(f"\n{'='*70}\nDIRECT CDSE REQUERY WITH COVERAGE COMPUTATION\n{'='*70}")
import sys
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))
import ms_scene_discovery, importlib
importlib.reload(ms_scene_discovery)
from ms_scene_discovery import calculate_boundary_coverage, search_sentinel2_directional
import geopandas as gpd
from datetime import datetime, timedelta

# helper: load aoi_bbox geometry (4326) for a city
def aoi_bbox_geom_4326(city):
    gdf = gpd.read_file(STACK_ROOT / city / "AOI.geojson")
    sub = gdf[gdf['feature_type'] == 'aoi_bbox']
    return sub.to_crs(4326).geometry.iloc[0]

# run a single query per city around battle start with relaxed cloud (25%) + wide window
# to see how many candidates exist at all, and their coverage %
for city in TARGETS:
    bbox = aoi_bbox_geom_4326(city)
    bnds = bbox.bounds
    print(f"\n--- {city} ---")
    print(f"  aoi_bbox bounds (4326): {tuple(round(b,4) for b in bnds)}")

    bs = city_meta.get(city, {}).get('battle_start', '')
    try:
        anchor = datetime.strptime(bs[:10], '%Y-%m-%d')
    except Exception:
        anchor = datetime(2022, 2, 24)

    # pull many candidates: +-90 days, cloud <=25%, both directions
    scenes = search_sentinel2_directional(bbox, bnds, anchor, window_days=90, cloud_threshold=25, direction='both')
    if not scenes:
        print(f"  CDSE returned 0 scenes in +-90d around {anchor.date()} @ cloud<=25")
        continue

    # compute coverage for each, tally how many pass the 90% gate
    print(f"  CDSE returned {len(scenes)} candidates (+-90d, cloud<=25%)")
    pass_90 = 0
    rows = []
    for s in scenes:
        fp = s.get('footprint') or s.get('geofootprint')
        cov = calculate_boundary_coverage(fp, bbox) if fp else float('nan')
        tile = s.get('tile_id') or '?'
        cc = s.get('cloud_cover')
        dt = s.get('date', '')[:10]
        if cov >= 90:
            pass_90 += 1
        rows.append((dt, tile, cc, cov))
    rows.sort()
    print(f"  >=90% coverage: {pass_90} / {len(scenes)}")
    print(f"  {'date':10s} {'tile':8s} {'cloud%':>7s} {'cov%':>7s}")
    for dt, tile, cc, cov in rows[:20]:
        print(f"  {dt:10s} {tile:8s} {cc if cc is not None else '?':>7}  {cov:>6.2f}")
    if len(rows) > 20:
        print(f"  ... ({len(rows)-20} more)")

    # per-tile coverage summary -- shows the tile-straddle clearly
    from collections import defaultdict
    by_tile = defaultdict(list)
    for dt, tile, cc, cov in rows:
        by_tile[tile].append(cov)
    print(f"  per-tile coverage:")
    for tile, covs in sorted(by_tile.items()):
        print(f"    {tile}: n={len(covs)}, cov range {min(covs):.1f}% -- {max(covs):.1f}%")

PER-CITY MS METADATA FILE CONTENTS

--- Borodyanka (0.6 KB) ---
  top keys: ['city', 'tier', 'battle_start', 'battle_end', 'conflict_ongoing', 'sar_aligned', 'pre_window', 'post_window', 'total_scenes', 'timestamp']
{
  "city": "Borodyanka",
  "tier": 1,
  "battle_start": "2022-02-28T00:00:00",
  "battle_end": "2022-04-01T00:00:00",
  "conflict_ongoing": false,
  "sar_aligned": true,
  "pre_window": {
    "label": "sar_aligned",
    "sar_dates": [
      "2020-12-31",
      "2021-01-06"
    ],
    "cloud_threshold_used": 15.0,
    "scenes_found": 0,
    "scenes": []
  },
  "post_window": {
    "label": "sar_aligned",
    "sar_dates": [
      "2022-05-07",
      "2022-05-19"
    ],
    "cloud_threshold_used": 15.0,
    "scenes_found": 0,
    "scenes": []
  },
  "total_scenes": 0,
  "timestamp": "2026-04-08T03:41:12.582899"
}

--- Bucha (7.4 KB) ---
  top keys: ['city', 'tier', 'battle_start', 'battle_end', 'conflict_ongoing', 'sar_aligned', 'pre_window', 'post_window', 'total_scenes', 't

In [43]:
# @title DIAGNOSTIC: bypass coverage filter and show per-scene coverage + tile_id
import sys
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))
import ms_scene_discovery, importlib
importlib.reload(ms_scene_discovery)
from ms_scene_discovery import calculate_boundary_coverage
import geopandas as gpd
import requests
from datetime import datetime, timedelta
import re as _re
from collections import defaultdict

def aoi_bbox_geom_4326(city):
    gdf = gpd.read_file(STACK_ROOT / city / "AOI.geojson")
    sub = gdf[gdf['feature_type'] == 'aoi_bbox']
    return sub.to_crs(4326).geometry.iloc[0]

def query_cdse_raw(bnds, start, end, cloud_max=50):
    """Same CDSE query but no coverage filter, return all hits with tile+coverage."""
    minx, miny, maxx, maxy = bnds
    wkt = f"POLYGON(({minx} {miny},{maxx} {miny},{maxx} {maxy},{minx} {maxy},{minx} {miny}))"
    filt = " and ".join([
        "Collection/Name eq 'SENTINEL-2'",
        f"OData.CSC.Intersects(area=geography'SRID=4326;{wkt}')",
        f"ContentDate/Start ge {start.strftime('%Y-%m-%d')}T00:00:00.000Z",
        f"ContentDate/Start le {end.strftime('%Y-%m-%d')}T23:59:59.999Z",
        "contains(Name,'L2A')",
    ])
    r = requests.get("https://catalogue.dataspace.copernicus.eu/odata/v1/Products",
                     params={'$filter': filt, '$orderby': 'ContentDate/Start desc',
                             '$top': 100, '$expand': 'Attributes'}, timeout=120)
    if r.status_code != 200:
        return None
    return r.json().get('value', [])

for city in ['Borodyanka', 'Bucha', 'Mykolaiv', 'Kharkiv']:
    bbox = aoi_bbox_geom_4326(city)
    bnds = bbox.bounds
    bs = city_meta.get(city, {}).get('battle_start', '2022-02-24')
    anchor = datetime.strptime(bs[:10], '%Y-%m-%d')
    start = anchor - timedelta(days=60)
    end   = anchor + timedelta(days=60)

    print(f"\n=== {city}  bbox {tuple(round(b,4) for b in bnds)} ===")
    raw = query_cdse_raw(bnds, start, end, cloud_max=50)
    if raw is None:
        print("  CDSE HTTP error"); continue
    print(f"  CDSE Intersects returned: {len(raw)} raw products")

    per_tile = defaultdict(list)
    for p in raw:
        name = p.get('Name', '')
        m = _re.search(r'_T(\d{2}[A-Z]{3})_', name)
        tile = m.group(1) if m else '?'
        cc = next((a['Value'] for a in p.get('Attributes', []) if a.get('Name') == 'cloudCover'), None)
        gf = p.get('GeoFootprint', {})
        fp = gf if isinstance(gf, dict) and gf.get('type') else (gf.get('Geography') if isinstance(gf, dict) else None)
        cov = calculate_boundary_coverage(fp, bbox) if fp else float('nan')
        per_tile[tile].append((p['ContentDate']['Start'][:10], cc, cov))

    for tile, rows in sorted(per_tile.items()):
        rows.sort()
        covs = [r[2] for r in rows if r[2] == r[2]]  # drop nan
        n_pass_90 = sum(1 for c in covs if c >= 90)
        n_pass_50 = sum(1 for c in covs if c >= 50)
        if covs:
            print(f"  {tile:8s}  n={len(rows):3d}  cov {min(covs):5.1f}-{max(covs):5.1f}%  "
                  f"pass@90: {n_pass_90}/{len(rows)}   pass@50: {n_pass_50}/{len(rows)}")
        else:
            print(f"  {tile:8s}  n={len(rows):3d}  (coverage calc failed)")


=== Borodyanka  bbox (29.6252, 50.5415, 30.0747, 50.8051) ===
  CDSE Intersects returned: 98 raw products
  35UPS     n= 49  cov  77.7- 77.7%  pass@90: 0/49   pass@50: 49/49
  35UQS     n= 49  cov  54.4- 54.4%  pass@90: 0/49   pass@50: 49/49

=== Bucha  bbox (30.154, 50.5096, 30.2663, 50.5811) ===
  CDSE Intersects returned: 100 raw products
  35UQS     n= 34  cov 100.0-100.0%  pass@90: 34/34   pass@50: 34/34
  36UUA     n= 33  cov   9.1-  9.1%  pass@90: 0/33   pass@50: 0/33
  36UUB     n= 33  cov  80.8- 80.8%  pass@90: 0/33   pass@50: 33/33

=== Mykolaiv  bbox (31.8695, 46.8167, 32.1092, 47.0492) ===
  CDSE Intersects returned: 99 raw products
  36TVS     n= 50  cov   6.2- 56.4%  pass@90: 0/50   pass@50: 24/50
  36TVT     n= 49  cov  21.7- 81.8%  pass@90: 0/49   pass@50: 24/49

=== Kharkiv  bbox (36.0934, 49.8764, 36.4617, 50.0918) ===
  CDSE Intersects returned: 48 raw products
  36UYA     n= 24  cov 100.0-100.0%  pass@90: 24/24   pass@50: 24/24
  37UCR     n= 24  cov  69.1- 69.1%  